# WARP for Decoder-only Models — Multi-Model × {SST-2, RTE} Pipeline

1. Run §1–§10 top to bottom (defines everything, doesn't start experiments).
2. Full run: call `run_warp_matrix()` under "Stage 1" (baselines in "Stage 2") — iterates `RUN_MATRIX`, auto-saves each group, prints a winners summary at the end.
3. Single group / debug: use the manual-mode section (§11).
4. Disconnects are safe: every stage (task-FT weights, GSN-FT weights, cache, QP checkpoints, results) is saved to Drive; rerunning resumes automatically.


### Before running

**Runtime → Restart session**, then run top to bottom. `CUBLAS_WORKSPACE_CONFIG` must take effect before the first CUDA call — the version-check cell asserts this.

### Just want to see finished results?

If you unzipped/uploaded the whole repo and opened this notebook directly: the "Mount Drive + paths" cell in §1 auto-detects the repo (`REPO_ROOT`), reads `data/dev_ann.json`/`test_ann.json` from it, and syncs `decoder/results_bundled/result_*.pkl` into the working folder — no manual Drive setup needed. Run through §10, then call `print_summary()` directly.

### Two-stage execution

1. **Stage 1** `run_warp_matrix()` — six WARP groups, no baselines
2. **Stage 2** `run_baseline_matrix()` — optional, adds LoRA / Full FT
3. **Summary** `print_summary()` — anytime; prints WARP only if baselines are missing


Run order: top to bottom through §12, then "Stage 1 — WARP" → "Stage 2 — Baselines (optional)" → "Summary". Pre-merge source files are in `_originals/`.


## §1. Install & imports

In [ ]:
# Install packages
!pip install -q "transformers>=4.44" datasets accelerate osqp scipy

!pip uninstall -y torchvision -q

try:
    import osqp
    _ = osqp.OSQP(algebra='cuda')
    HAS_CUDA_OSQP = True
    print('CUDA OSQP available.')
except Exception:
    HAS_CUDA_OSQP = False
    print('CUDA OSQP not available → will use CPU (builtin) algebra.')

In [ ]:
# Imports + deterministic settings
import os

BLAS_THREADS = '1'

os.environ.setdefault('PYTHONHASHSEED', '0')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('OMP_NUM_THREADS', BLAS_THREADS)
os.environ.setdefault('MKL_NUM_THREADS', BLAS_THREADS)
os.environ.setdefault('OPENBLAS_NUM_THREADS', BLAS_THREADS)
os.environ.setdefault('NUMEXPR_NUM_THREADS', BLAS_THREADS)

import json, time, copy, pickle, gc, random, hashlib, platform
from collections import defaultdict

import numpy as np
import scipy
import scipy.sparse as sp
from scipy.sparse.linalg import svds

import torch
import torch.nn as nn
import torch.nn.functional as F

import osqp
import transformers, datasets
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          GPT2ForSequenceClassification)

SEED   = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.backends.cudnn.deterministic    = True
torch.backends.cudnn.benchmark        = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32       = False
torch.set_float32_matmul_precision('highest')
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as _e:
    print(f'Warning: use_deterministic_algorithms unavailable: {_e}')

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def stage_seed(model_key, task, stage, base=SEED):
    h = hashlib.md5(f'{base}|{model_key}|{task}|{stage}'.encode()).hexdigest()
    return int(h[:8], 16) % (2 ** 31)

def arr_hash(x):
    return hashlib.sha256(
        np.ascontiguousarray(x, dtype=np.float64).tobytes()).hexdigest()[:12]

set_all_seeds(SEED)

ENV_FINGERPRINT = {
    'python':       platform.python_version(),
    'torch':        torch.__version__,
    'transformers': transformers.__version__,
    'datasets':     datasets.__version__,
    'numpy':        np.__version__,
    'scipy':        scipy.__version__,
    'osqp':         getattr(osqp, '__version__', 'unknown'),
    'gpu':          (torch.cuda.get_device_name(0)
                     if torch.cuda.is_available() else 'cpu'),
}
print(f'device={DEVICE}')
for _k, _v in ENV_FINGERPRINT.items():
    print(f'  {_k:<13}= {_v}')
print('\nBit-identical reproducibility is only guaranteed on the same machine / GPU / package versions.')
print('  Across GPU architectures (T4 vs L4 vs A100), cuBLAS picks different kernels — not guaranteed.')
print('  Paper-level approach: treat W(0) and the cache as release artifacts (with arr_hash);')
print('  everything after W(0) runs in numpy+OSQP float64, which is bit-reproducible on any machine.')

In [ ]:
# Mount Drive + resolve paths
REPO_SUBDIR = 'code'

_REPO_MARKERS = ['decoder/decoder_experiments.ipynb', 'decoder/results_bundled',
                 'data/dev_ann.json']

def _has_repo_marker(d):
    return any(os.path.exists(f'{d}/{m}') for m in _REPO_MARKERS)

def _extract_zip_once(zpath):
    dest = zpath[:-4] if zpath.lower().endswith('.zip') else zpath + '_extracted'
    if os.path.isdir(dest):
        return dest
    import zipfile
    try:
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(dest)
        print(f'[repo] auto-extracted {os.path.basename(zpath)} -> {dest}')
        return dest
    except Exception as e:
        print(f'[repo] could not extract {zpath}: {e}')
        return None

def _find_repo_root(cands):
    # 1) direct folder hit, or one level inside it (zip had an extra wrapping dir)
    search = []
    for c in cands:
        search.append(c)
        if os.path.isdir(c):
            search += [f'{c}/{sub}' for sub in os.listdir(c)]
    for d in search:
        if _has_repo_marker(d):
            return os.path.abspath(d)
    # 2) nothing found as a folder yet -> look for an unextracted .zip and try that
    import glob
    zip_dirs = {os.path.dirname(c) for c in cands if os.path.isdir(os.path.dirname(c))}
    for zd in zip_dirs:
        for zpath in glob.glob(f'{zd}/*.zip'):
            extracted = _extract_zip_once(zpath)
            if not extracted:
                continue
            if _has_repo_marker(extracted):
                return os.path.abspath(extracted)
            if os.path.isdir(extracted):
                for sub in os.listdir(extracted):
                    if _has_repo_marker(f'{extracted}/{sub}'):
                        return os.path.abspath(f'{extracted}/{sub}')
    return None

DRIVE_SUBDIR = 'WARP Stuff'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = f'/content/drive/MyDrive/{DRIVE_SUBDIR}'
    _repo_cands = [f'/content/drive/MyDrive/{REPO_SUBDIR}', f'/content/{REPO_SUBDIR}']
except Exception:
    BASE = f'./{DRIVE_SUBDIR}'
    print('Not on Colab → using local dir', BASE)
    _repo_cands = [f'./{REPO_SUBDIR}', '.', '..']

os.makedirs(BASE, exist_ok=True)

REPO_ROOT = _find_repo_root(_repo_cands)
print(f'[repo] found repo at → {REPO_ROOT}' if REPO_ROOT else
      f'Warning: repo not found (tried {_repo_cands}). '
      f'dev_ann/test_ann and bundled results will depend on files manually placed in Drive. '
      f'If you did unzip/upload the repo, check that REPO_SUBDIR matches the folder name.')

def _find_first(*cands):
    for p in cands:
        if os.path.exists(p):
            return p
    print(f'Warning: annotation file not found, tried: {list(cands)}')
    return cands[0]

DEV_PATH  = _find_first(
    *([f'{REPO_ROOT}/data/dev_ann.json'] if REPO_ROOT else []),
    f'{BASE}/dev_ann.json', f'{BASE}/adv_glue/dev_ann.json')
TEST_PATH = _find_first(
    *([f'{REPO_ROOT}/data/test_ann.json'] if REPO_ROOT else []),
    f'{BASE}/test_ann.json', f'{BASE}/adv_glue/test_ann.json', f'{BASE}/adv_glue/test.json')

RESULT_DIR = f'{BASE}/results_decoder'
CKPT_DIR   = f'{BASE}/ckpt_decoder'
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

def _walk_limited(root, maxdepth=4):
    # os.walk capped at maxdepth levels below root, so this never scans
    # an entire Drive/filesystem — just enough to see through a few
    # nested/renamed folders (e.g. "results_bundled/results_bundled copy/").
    root = root.rstrip('/')
    if not os.path.isdir(root):
        return
    base_depth = root.count('/')
    for dirpath, dirnames, filenames in os.walk(root):
        yield dirpath, dirnames, filenames
        if dirpath.count('/') - base_depth >= maxdepth:
            dirnames[:] = []

def _extract_all_zips_under(root, maxdepth=2):
    for dirpath, dirnames, filenames in list(_walk_limited(root, maxdepth)):
        for fn in filenames:
            if fn.lower().endswith('.zip'):
                _extract_zip_once(f'{dirpath}/{fn}')

def _find_bundled_results_dir(roots):
    # Finds any folder containing result_*.pkl under the given roots,
    # regardless of naming/nesting; auto-extracts any .zip it passes along the way.
    for root in roots:
        _extract_all_zips_under(root)
    for root in roots:
        for dirpath, dirnames, filenames in _walk_limited(root):
            if any(fn.startswith('result_') and fn.endswith('.pkl') for fn in filenames):
                return dirpath
    return None

_search_roots = list(dict.fromkeys(
    [os.path.dirname(c) for c in _repo_cands if os.path.isdir(os.path.dirname(c))]
    + _repo_cands))
_bundled_dir = ((f'{REPO_ROOT}/decoder/results_bundled' if REPO_ROOT
                and os.path.isdir(f'{REPO_ROOT}/decoder/results_bundled') else None)
               or _find_bundled_results_dir(_search_roots))
if _bundled_dir:
    import shutil
    _synced = [f for f in os.listdir(_bundled_dir)
               if f.startswith('result_') and f.endswith('.pkl')
               and not os.path.exists(f'{RESULT_DIR}/{f}')]
    for _fn in _synced:
        shutil.copy(f'{_bundled_dir}/{_fn}', f'{RESULT_DIR}/{_fn}')
    print(f'[bundled] found results at {_bundled_dir}, synced {len(_synced)} new pkl(s) → {RESULT_DIR}'
          if _synced else f'[bundled] found results at {_bundled_dir}, nothing new to sync')
else:
    print('[bundled] no results_bundled folder or zip found — skipping sync')

print(f'dev_ann  -> {DEV_PATH}')
print(f'test_ann -> {TEST_PATH}')
print('paths ok:', RESULT_DIR)

## §2. Global config: RUN_MATRIX / MODEL_REGISTRY / task config / hyperparameters

**Only this section** needs editing to control which (model, task) combinations run.

In [ ]:
# MODEL_REGISTRY + RUN_MATRIX
MODEL_REGISTRY = {
    'gpt2': {
        'family':   'gpt2',
        'hf_id':    'gpt2',
        'task_ckpt': {
            'sst2': 'PavanNeerudu/gpt2-finetuned-sst2',
            'rte':  None,
        },
        'qp': dict(RANK=64, MARGIN_S=0.1, MARGIN_H=0.3, LAMBDA=30.0, REG=20.0)
    },
    'gpt2-medium': {
        'family':   'gpt2',
        'hf_id':    'gpt2-medium',
        'task_ckpt': {'sst2': None, 'rte': None},
        'qp': dict(RANK=32, MARGIN_S=0.1, MARGIN_H=0.3, LAMBDA=10.0, REG=20.0),
    },
    'distilgpt2': {
        'family':   'gpt2',
        'hf_id':    'distilgpt2',
        'task_ckpt': {'sst2': None, 'rte': None},
        'qp': dict(RANK=32, MARGIN_S=0.1, MARGIN_H=0.3, LAMBDA=10.0, REG=20.0),
    },
    'smollm2-360m': {
        'family':   'gated',
        'hf_id':    'HuggingFaceTB/SmolLM2-360M',
        'task_ckpt': {'sst2': None, 'rte': None},
        'qp': dict(RANK=64, MARGIN_S=0.5, MARGIN_H=0.3, LAMBDA=10.0, REG=5.0),
    },
    'qwen2.5-0.5b': {
        'family':   'gated',
        'hf_id':    'Qwen/Qwen2.5-0.5B',
        'task_ckpt': {'sst2': None, 'rte': None},
        'qp': dict(RANK=64, MARGIN_S=0.5, MARGIN_H=0.3, LAMBDA=10.0, REG=5.0),
    },
    'qwen2.5-1.5b': {
        'family':   'gated',
        'hf_id':    'Qwen/Qwen2.5-1.5B',
        'task_ckpt': {'sst2': None, 'rte': None},
        'qp': dict(RANK=64, MARGIN_S=0.5, MARGIN_H=0.3, LAMBDA=10.0, REG=5.0),
    },
    'pythia-160m': {
        'family':   'neox',
        'hf_id':    'EleutherAI/pythia-160m',
        'task_ckpt': {'sst2': None, 'rte': None},
        'qp': dict(RANK=32, MARGIN_S=0.5, MARGIN_H=0.3, LAMBDA=10.0, REG=20.0),
    },
    'pythia-410m': {
        'family':   'neox',
        'hf_id':    'EleutherAI/pythia-410m',
        'task_ckpt': {'sst2': None, 'rte': None},
        'qp': dict(RANK=32, MARGIN_S=0.5, MARGIN_H=0.3, LAMBDA=10.0, REG=20.0),
    },
    'gpt2-large': {
        'family':   'gpt2',
        'hf_id':    'gpt2-large',
        'task_ckpt': {'sst2': None, 'rte': None},
        'qp': dict(RANK=32, MARGIN_S=0.1, MARGIN_H=0.3, LAMBDA=10.0, REG=20.0),
    },
    'tinyllama-1.1b': {
        'family':   'gated',
        'hf_id':    'TinyLlama/TinyLlama_v1.1',
        'task_ckpt': {'sst2': None, 'rte': None},
        'qp': dict(RANK=64, MARGIN_S=0.5, MARGIN_H=0.3, LAMBDA=10.0, REG=5.0),
    },
    'llama-3.2-1b': {
        'family':   'gated',
        'hf_id':    'meta-llama/Llama-3.2-1B',
        'task_ckpt': {'sst2': None, 'rte': None},
        'qp': dict(RANK=64, MARGIN_S=0.5, MARGIN_H=0.3, LAMBDA=10.0, REG=5.0),
    },
}

RUN_MATRIX = [
    ('gpt2',         'sst2'),
    ('gpt2',         'rte'),
    ('gpt2-medium',  'sst2'),
    ('gpt2-medium',  'rte'),
    ('qwen2.5-0.5b', 'sst2'),
    ('qwen2.5-0.5b', 'rte'),
]

RUN_BASELINES = True
RUN_ABLATIONS = False

In [ ]:
# Task config + hyperparameters
def _fmt_sst2(item):
    return item['sentence']

def _fmt_rte(item):
    return f"premise: {item['sentence1']}\nhypothesis: {item['sentence2']}"

TASK_CONFIG = {
    'sst2': {
        'advglue_key':  'sst2',
        'fmt':          _fmt_sst2,
        'n_remain':     200,
        'n_clean':      2000,
        'ft_epochs':    3, 'ft_lr': 2e-5, 'ft_bs': 16, 'max_len': 128,
    },
    'rte': {
        'advglue_key':  'rte',
        'fmt':          _fmt_rte,
        'n_remain':     200,
        'n_clean':      500,
        'ft_epochs':    5, 'ft_lr': 2e-5, 'ft_bs': 8, 'max_len': 256,
    },
}

EXCLUDED_METHODS = {'AdvFever', 'BERT-ATTACK', 'SCPN', 'SemAttack'}

REPAIR_GAP_FLOOR_MODE = 'relative'
REPAIR_GAP_FLOOR_REL  = 1.5
REPAIR_GAP_FLOOR_ABS  = -15.0

N_REMAIN_QP = 200

LORA_MATCH_DOF = True

MAX_FT_SAMPLES = 20000

MAX_ITER  = 300
GAP_TOL   = 1e-4
QP_PATIENCE = 25
SAVE_EVERY  = 10

KAPPA_TARGET = 1.0
FT_MAX_STEPS = 30
FT_LR        = 1e-3
N_AUX        = 8

BASELINE_N_SEEDS = 1

BASELINE_LR = 1e-4
BASELINE_BS = 16
BASELINE_MAX_EPOCHS = 300
BASELINE_PATIENCE   = 5
BASELINE_TOL        = 1e-6

In [ ]:
# Run-mode settings
ALGO_TAG = 'paper-faithful-alg1'

RUN_MODE = 'resume'

FORCE_BUILTIN_OSQP = True

print(f'ALGO={ALGO_TAG}  RUN_MODE={RUN_MODE}  '
      f'FORCE_BUILTIN_OSQP={FORCE_BUILTIN_OSQP}')
print(f'RUN_MATRIX ({len(RUN_MATRIX)} groups):')
for _mk, _tk in RUN_MATRIX:
    _hp = {**MODEL_REGISTRY[_mk]['qp'],
           **MODEL_REGISTRY[_mk].get('qp_task', {}).get(_tk, {})}
    print(f'   {_mk:<14} x {_tk:<5}  HP={_hp}')

In [ ]:
# GSN probes + GSN-FT aux set
GSN_PROBES = {
    'sst2': [
        'this movie is terrible and boring',
        'i really enjoyed this film',
        'the acting was awful',
        'a wonderful and moving story',
        'complete waste of time',
        'absolutely brilliant performance',
        'i fell asleep halfway through',
        'the best film i have seen this year',
    ],
    'rte': [
        'premise: A man is playing a guitar on stage.\nhypothesis: A man is performing music.',
        'premise: The cat sat on the mat.\nhypothesis: A dog is running in the park.',
        'premise: Scientists discovered a new species of frog in the rainforest.\nhypothesis: A new animal species was found.',
        'premise: The company reported record profits this quarter.\nhypothesis: The company lost money this quarter.',
        'premise: Children were playing soccer in the schoolyard.\nhypothesis: Kids were engaged in a sport.',
        'premise: The museum opens at nine in the morning.\nhypothesis: The museum is closed all day.',
        'premise: She graduated from Harvard with a degree in physics.\nhypothesis: She has a university degree.',
        'premise: The train arrived two hours late due to snow.\nhypothesis: The train was on time.',
    ],
}

AUX_SAMPLES = {
    'sst2': [
        ('an absorbing and beautifully shot film',                   1),
        ('the performances are excellent and the script is sharp',   1),
        ('a moving , thoughtful and ultimately uplifting story',     1),
        ('genuinely funny and surprisingly heartfelt',               1),
        ('a tedious , poorly written waste of two hours',            0),
        ('thoroughly boring with wooden acting and a clichéd plot',  0),
        ('an utter mess of a film with no redeeming qualities',      0),
        ('painful to sit through ; i can not recommend it',          0),
    ],
    'rte': [
        ('premise: The chef prepared a delicious three-course meal for the guests.\nhypothesis: The chef cooked food.',              0),
        ('premise: Heavy rain caused flooding in several parts of the city.\nhypothesis: The city experienced flooding.',            0),
        ('premise: The orchestra performed a symphony by Beethoven last night.\nhypothesis: Music was performed last night.',        0),
        ('premise: The marathon runner finished the race in under three hours.\nhypothesis: The runner completed the marathon.',     0),
        ('premise: The library is closed on national holidays.\nhypothesis: The library is open every day of the year.',             1),
        ('premise: The professor cancelled the lecture because she was ill.\nhypothesis: The lecture took place as scheduled.',      1),
        ('premise: The bakery sells only bread and pastries.\nhypothesis: The bakery sells fresh seafood.',                          1),
        ('premise: The flight to Tokyo departs from gate twelve.\nhypothesis: The flight to Tokyo was cancelled.',                   1),
    ],
}

FIXED_V0_STORE = {}
print('Config defined.')

## §3. Architecture layer: numerical primitives + family dispatch + adapters

In [ ]:
# Numerical primitives (activations, norms)
_SQRT_2_OVER_PI = float(np.sqrt(2.0 / np.pi))

def act_gelu(x):
    inner = _SQRT_2_OVER_PI * (x + 0.044715 * x**3)
    return 0.5 * x * (1.0 + np.tanh(inner))

def dact_gelu(x):
    inner   = _SQRT_2_OVER_PI * (x + 0.044715 * x**3)
    th      = np.tanh(inner)
    d_inner = _SQRT_2_OVER_PI * (1.0 + 3.0 * 0.044715 * x**2)
    return 0.5 * (1.0 + th) + 0.5 * x * (1.0 - th**2) * d_inner

from scipy.special import erf as _erf

def act_gelu_erf(x):
    return 0.5 * x * (1.0 + _erf(x / np.sqrt(2.0)))

def dact_gelu_erf(x):
    Phi = 0.5 * (1.0 + _erf(x / np.sqrt(2.0)))
    phi = np.exp(-0.5 * x * x) / np.sqrt(2.0 * np.pi)
    return Phi + x * phi

def sigmoid_np(x):
    return np.where(x >= 0, 1.0 / (1.0 + np.exp(-x)),
                    np.exp(x) / (1.0 + np.exp(x)))

def act_silu(x):
    return x * sigmoid_np(x)

def dact_silu(x):
    sig = sigmoid_np(x)
    return sig * (1.0 + x * (1.0 - sig))

def layer_norm_np(x, gamma, beta, eps):
    mu  = float(np.mean(x))
    var = float(np.mean((x - mu) ** 2))
    return gamma * (x - mu) / np.sqrt(var + eps) + beta

def rms_norm_np(x, gamma, eps):
    rms = np.sqrt(np.mean(x ** 2) + eps)
    return gamma * x / rms

In [ ]:
# Family dispatch (forward / backward / Lipschitz)
def _gpt2_forward_gap(W, v, ctx):
    pre   = W @ v + ctx['b_fc']
    h     = act_gelu(pre)
    out_h = ctx['wo'] @ h + ctx['b_proj']
    y_blk = ctx['res'] + out_h
    y_ln  = layer_norm_np(y_blk, ctx['gamma_ln'], ctx['beta_ln'], ctx['eps_ln'])
    gap   = float(ctx['w_diff'] @ y_ln + ctx['b_diff'])
    return gap

def _gpt2_dgap_dpre(W, v, ctx):
    pre  = W @ v + ctx['b_fc']
    h    = act_gelu(pre)
    y    = ctx['res'] + ctx['wo'] @ h + ctx['b_proj']
    mu   = float(np.mean(y)); z = y - mu; D = y.shape[0]
    var  = float(np.mean(z * z)); sigma = np.sqrt(var + ctx['eps_ln'])
    c    = ctx['w_diff'] * ctx['gamma_ln']
    S    = float(np.dot(c, z))
    dg_dy = (c - float(np.mean(c))) / sigma - (S * z) / (D * sigma**3)
    dg_dh = ctx['wo'].T @ dg_dy
    return dg_dh * dact_gelu(pre)

def _gpt2_lip_sound(W, ctx):
    ACT_LIP = 1.13
    lip_ln  = float(np.max(np.abs(ctx['gamma_ln']))) / np.sqrt(ctx['eps_ln'])
    return (float(np.linalg.norm(ctx['w_diff'])) * lip_ln *
            float(np.linalg.norm(ctx['wo'], ord=2)) * ACT_LIP *
            float(np.linalg.norm(W, ord=2)))

def _gated_forward_gap(W, v, ctx):
    gate_pre = W @ v + ctx['b_gate']
    up_pre   = ctx['Wu'] @ v + ctx['b_up']
    hidden   = act_silu(gate_pre) * up_pre
    out_mlp  = ctx['Wd'] @ hidden + ctx['b_down']
    y_blk    = ctx['res'] + out_mlp
    y_rms    = rms_norm_np(y_blk, ctx['gamma_ln'], ctx['eps_ln'])
    return float(ctx['w_diff'] @ y_rms + ctx['b_diff'])

def _gated_dgap_dpre(W, v, ctx):
    gate_pre = W @ v + ctx['b_gate']
    up_pre   = ctx['Wu'] @ v + ctx['b_up']
    hidden   = act_silu(gate_pre) * up_pre
    y_blk    = ctx['res'] + ctx['Wd'] @ hidden + ctx['b_down']
    D    = y_blk.shape[0]
    rms  = np.sqrt(np.mean(y_blk ** 2) + ctx['eps_ln'])
    c    = ctx['w_diff'] * ctx['gamma_ln']
    S    = float(np.dot(c, y_blk))
    dg_dy = c / rms - (S * y_blk) / (D * rms ** 3)
    dg_dh = ctx['Wd'].T @ dg_dy
    dg_dgate_act = dg_dh * up_pre
    return dg_dgate_act * dact_silu(gate_pre)

def _gated_lip_sound(W, ctx):
    ACT_LIP = 1.1
    lip_rms = float(np.max(np.abs(ctx['gamma_ln']))) / np.sqrt(ctx['eps_ln'])
    return (float(np.linalg.norm(ctx['w_diff'])) * lip_rms *
            float(np.linalg.norm(ctx['Wd'], ord=2)) * ACT_LIP *
            float(np.linalg.norm(ctx['Wu'], ord=2)) *
            float(np.linalg.norm(W, ord=2)))

def _gated_eps_star_factory(W, ctx):
    C   = (float(np.linalg.norm(ctx['w_diff']))
           * float(np.max(np.abs(ctx['gamma_ln']))) / np.sqrt(ctx['eps_ln'])
           * float(np.linalg.norm(ctx['Wd'], ord=2)))
    nW  = float(np.linalg.norm(W, ord=2))
    nWu = float(np.linalg.norm(ctx['Wu'], ord=2))
    a   = C * (1.1 + 1.0) * nW * nWu
    def eps_star(v, gap):
        if gap <= 0: return 0.0
        u_inf = float(np.max(np.abs(ctx['Wu'] @ v + ctx['b_up'])))
        g_inf = float(np.max(np.abs(W @ v + ctx['b_gate'])))
        b = C * (1.1 * u_inf * nW + (g_inf + 0.28) * nWu)
        return float((-b + np.sqrt(b * b + 2.0 * a * gap)) / (2.0 * a))
    return eps_star

def _neox_forward_gap(W, v, ctx):
    pre   = W @ v + ctx['b_fc']
    h     = act_gelu_erf(pre)
    y_blk = ctx['res'] + ctx['wo'] @ h + ctx['b_proj']
    y_ln  = layer_norm_np(y_blk, ctx['gamma_ln'], ctx['beta_ln'], ctx['eps_ln'])
    return float(ctx['w_diff'] @ y_ln + ctx['b_diff'])

def _neox_dgap_dpre(W, v, ctx):
    pre  = W @ v + ctx['b_fc']
    h    = act_gelu_erf(pre)
    y    = ctx['res'] + ctx['wo'] @ h + ctx['b_proj']
    mu   = float(np.mean(y)); z = y - mu; D = y.shape[0]
    var  = float(np.mean(z * z)); sigma = np.sqrt(var + ctx['eps_ln'])
    c    = ctx['w_diff'] * ctx['gamma_ln']
    S    = float(np.dot(c, z))
    dg_dy = (c - float(np.mean(c))) / sigma - (S * z) / (D * sigma**3)
    dg_dh = ctx['wo'].T @ dg_dy
    return dg_dh * dact_gelu_erf(pre)

def _neox_lip_sound(W, ctx):
    ACT_LIP = 1.13
    lip_ln  = float(np.max(np.abs(ctx['gamma_ln']))) / np.sqrt(ctx['eps_ln'])
    return (float(np.linalg.norm(ctx['w_diff'])) * lip_ln *
            float(np.linalg.norm(ctx['wo'], ord=2)) * ACT_LIP *
            float(np.linalg.norm(W, ord=2)))

FAMILY_FUNCS = {
    'neox':  {'forward': _neox_forward_gap,  'dpre': _neox_dgap_dpre,  'lip': _neox_lip_sound},
    'gpt2':  {'forward': _gpt2_forward_gap,  'dpre': _gpt2_dgap_dpre,  'lip': _gpt2_lip_sound},
    'gated': {'forward': _gated_forward_gap, 'dpre': _gated_dgap_dpre, 'lip': _gated_lip_sound},
}

def forward_gap_np(W, v, ctx):
    return FAMILY_FUNCS[ctx['fam']]['forward'](W, v, ctx)

def dgap_dpre(W, v, ctx):
    return FAMILY_FUNCS[ctx['fam']]['dpre'](W, v, ctx)

def lipschitz_sound(W, ctx):
    return FAMILY_FUNCS[ctx['fam']]['lip'](W, ctx)

def predict_w(v, W, ctx):
    return 1 if forward_gap_np(W, v, ctx) > 0 else 0

def gap_w(v, tgt, W, ctx):
    g = forward_gap_np(W, v, ctx)
    return g if tgt == 1 else -g

def eval_acc(cache, W, use_gt=True):
    key = 'gt' if use_gt else 'pred'
    if not cache: return 0.0
    return 100.0 * sum(predict_w(s['v'], W, s['ctx']) == s[key] for s in cache) / len(cache)

def lipschitz_local(W, ctx, v, n_probes=8, h=1e-3):
    g0 = forward_gap_np(W, v, ctx)
    rng_ = np.random.default_rng(123)
    max_dd = 0.0
    for _ in range(n_probes):
        d = rng_.standard_normal(v.shape[0]); d /= np.linalg.norm(d)
        dd = abs(forward_gap_np(W, v + h * d, ctx) - g0) / h
        max_dd = max(max_dd, dd)
    return float(max_dd)

print('Numerical primitives + family dispatch defined.')

In [ ]:
# Architecture adapters (GPT-2 / gated / NeoX)
class GPT2Adapter:
    family = 'gpt2'

    def repair_module(self, model):
        return model.transformer.h[-1].mlp.c_fc

    def set_repair_module(self, model, mod):
        model.transformer.h[-1].mlp.c_fc = mod

    def weight_np(self, model):
        return (self.repair_module(model).weight
                .detach().cpu().numpy().astype(np.float64).T.copy())

    def build_head_static(self, model):
        blk  = model.transformer.h[-1]
        ln_f = model.transformer.ln_f
        Wsc  = model.score.weight.detach().cpu().numpy().astype(np.float64)
        b_diff = 0.0
        if model.score.bias is not None:
            b_sc = model.score.bias.detach().cpu().numpy().astype(np.float64)
            b_diff = float(b_sc[1] - b_sc[0])
        return {
            'fam':      self.family,
            'b_fc':     blk.mlp.c_fc.bias.detach().cpu().numpy().astype(np.float64).copy(),
            'wo':       blk.mlp.c_proj.weight.detach().cpu().numpy().astype(np.float64).T.copy(),
            'b_proj':   blk.mlp.c_proj.bias.detach().cpu().numpy().astype(np.float64).copy(),
            'gamma_ln': ln_f.weight.detach().cpu().numpy().astype(np.float64).copy(),
            'beta_ln':  ln_f.bias.detach().cpu().numpy().astype(np.float64).copy(),
            'eps_ln':   float(ln_f.eps),
            'w_diff':   (Wsc[1] - Wsc[0]).copy(),
            'b_diff':   b_diff,
        }

    def build_sample_ctx(self, model, tok, text, head_static, max_len=256):
        model.eval()
        blk = model.transformer.h[-1]
        cap = {}
        hs = [blk.mlp.c_fc.register_forward_hook(
                  lambda m, i, o: cap.__setitem__('v', i[0].detach().cpu())),
              blk.ln_2.register_forward_hook(
                  lambda m, i, o: cap.__setitem__('res', i[0].detach().cpu()))]
        enc = tok(text, return_tensors='pt', truncation=True, max_length=max_len)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        for h in hs: h.remove()
        t = enc['input_ids'].shape[1] - 1
        ctx = dict(head_static)
        ctx['res'] = cap['res'][0, t].numpy().astype(np.float64)
        v_np = cap['v'][0, t].numpy().astype(np.float64)
        logits = out.logits[0].detach().cpu().numpy()
        return v_np, ctx, (1 if logits[1] > logits[0] else 0)

class GatedAdapter:
    family = 'gated'

    def repair_module(self, model):
        return model.model.layers[-1].mlp.gate_proj

    def set_repair_module(self, model, mod):
        model.model.layers[-1].mlp.gate_proj = mod

    def weight_np(self, model):
        return (self.repair_module(model).weight
                .detach().cpu().numpy().astype(np.float64).copy())

    def build_head_static(self, model):
        blk = model.model.layers[-1]
        mlp = blk.mlp
        def _b(mod, n):
            return (mod.bias.detach().cpu().numpy().astype(np.float64).copy()
                    if mod.bias is not None else np.zeros(n))
        Wu = mlp.up_proj.weight.detach().cpu().numpy().astype(np.float64).copy()
        Wd = mlp.down_proj.weight.detach().cpu().numpy().astype(np.float64).copy()
        ln = model.model.norm
        Wsc = model.score.weight.detach().cpu().numpy().astype(np.float64)
        b_diff = 0.0
        if model.score.bias is not None:
            b_sc = model.score.bias.detach().cpu().numpy().astype(np.float64)
            b_diff = float(b_sc[1] - b_sc[0])
        return {
            'fam':      self.family,
            'Wu':       Wu, 'Wd': Wd,
            'b_gate':   _b(mlp.gate_proj, Wu.shape[0]),
            'b_up':     _b(mlp.up_proj,   Wu.shape[0]),
            'b_down':   _b(mlp.down_proj, Wd.shape[0]),
            'gamma_ln': ln.weight.detach().cpu().numpy().astype(np.float64).copy(),
            'eps_ln':   float(getattr(ln, 'variance_epsilon', 1e-6)),
            'w_diff':   (Wsc[1] - Wsc[0]).copy(),
            'b_diff':   b_diff,
        }

    def build_sample_ctx(self, model, tok, text, head_static, max_len=256):
        model.eval()
        blk = model.model.layers[-1]
        cap = {}
        hs = [blk.mlp.gate_proj.register_forward_hook(
                  lambda m, i, o: cap.__setitem__('v', i[0].detach().cpu())),
              blk.post_attention_layernorm.register_forward_hook(
                  lambda m, i, o: cap.__setitem__('res', i[0].detach().cpu()))]
        enc = tok(text, return_tensors='pt', truncation=True, max_length=max_len)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        for h in hs: h.remove()
        t = enc['input_ids'].shape[1] - 1
        ctx = dict(head_static)
        ctx['res'] = cap['res'][0, t].numpy().astype(np.float64)
        v_np = cap['v'][0, t].numpy().astype(np.float64)
        logits = out.logits[0].detach().cpu().numpy()
        return v_np, ctx, (1 if logits[1] > logits[0] else 0)

class NeoXAdapter:
    family = 'neox'

    def repair_module(self, model):
        return model.gpt_neox.layers[-1].mlp.dense_h_to_4h

    def set_repair_module(self, model, mod):
        model.gpt_neox.layers[-1].mlp.dense_h_to_4h = mod

    def weight_np(self, model):
        return (self.repair_module(model).weight
                .detach().cpu().numpy().astype(np.float64).copy())

    def build_head_static(self, model):
        blk  = model.gpt_neox.layers[-1]
        ln_f = model.gpt_neox.final_layer_norm
        Wsc  = model.score.weight.detach().cpu().numpy().astype(np.float64)
        b_diff = 0.0
        if model.score.bias is not None:
            b_sc = model.score.bias.detach().cpu().numpy().astype(np.float64)
            b_diff = float(b_sc[1] - b_sc[0])
        return {
            'fam':      self.family,
            'b_fc':     blk.mlp.dense_h_to_4h.bias.detach().cpu().numpy().astype(np.float64).copy(),
            'wo':       blk.mlp.dense_4h_to_h.weight.detach().cpu().numpy().astype(np.float64).copy(),
            'b_proj':   blk.mlp.dense_4h_to_h.bias.detach().cpu().numpy().astype(np.float64).copy(),
            'gamma_ln': ln_f.weight.detach().cpu().numpy().astype(np.float64).copy(),
            'beta_ln':  ln_f.bias.detach().cpu().numpy().astype(np.float64).copy(),
            'eps_ln':   float(ln_f.eps),
            'w_diff':   (Wsc[1] - Wsc[0]).copy(),
            'b_diff':   b_diff,
        }

    def build_sample_ctx(self, model, tok, text, head_static, max_len=256):
        model.eval()
        blk = model.gpt_neox.layers[-1]
        cap = {}
        hs = [blk.mlp.dense_h_to_4h.register_forward_hook(
                  lambda m, i, o: cap.__setitem__('v', i[0].detach().cpu())),
              blk.mlp.register_forward_hook(
                  lambda m, i, o: cap.__setitem__('mlp', o.detach().cpu())),
              blk.register_forward_hook(
                  lambda m, i, o: cap.__setitem__('blk', o[0].detach().cpu()))]
        enc = tok(text, return_tensors='pt', truncation=True, max_length=max_len)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
        for h in hs: h.remove()
        t = enc['input_ids'].shape[1] - 1
        ctx = dict(head_static)
        ctx['res'] = (cap['blk'][0, t] - cap['mlp'][0, t]).numpy().astype(np.float64)
        v_np = cap['v'][0, t].numpy().astype(np.float64)
        logits = out.logits[0].detach().cpu().numpy()
        return v_np, ctx, (1 if logits[1] > logits[0] else 0)

ADAPTERS = {'gpt2': GPT2Adapter(), 'gated': GatedAdapter(), 'neox': NeoXAdapter()}
print('Adapters defined.')

In [ ]:
# GSN (Gap Sensitivity Norm)
def _fixed_v0(key, dim):
    if key not in FIXED_V0_STORE:
        v0 = np.random.default_rng(42).standard_normal(dim)
        FIXED_V0_STORE[key] = v0 / np.linalg.norm(v0)
    return FIXED_V0_STORE[key][:dim]

def compute_gsn(adapter, model, tok, probes, head_static, v0_key,
                rank_probe=2, max_len=256):
    W = adapter.weight_np(model)
    d_out, d_in = W.shape
    k = min(rank_probe, min(d_out, d_in) - 1)
    U, _, _ = svds(W, k=k, v0=_fixed_v0(v0_key, min(d_out, d_in)))
    U = U.copy()
    norms = []
    for s in probes:
        v, ctx, _ = adapter.build_sample_ctx(model, tok, s, head_static, max_len)
        eff = dgap_dpre(W, v, ctx)
        coeffs = np.stack([float(U[:, r] @ eff) * v for r in range(k)])
        norms.append(float(np.linalg.norm(coeffs)))
    return float(np.mean(norms)), float(np.linalg.norm(W, ord=2))

print('GSN defined.')

## §4. Model prep: Stage-A task FT → Stage-B GSN-FT

In [ ]:
# Stage A: task fine-tune
def task_finetune(model, tok, task, tcfg, save_dir=None, done_epochs=0):
    print(f'[Stage A] full FT on GLUE/{task}: {tcfg["ft_epochs"]} epochs, lr={tcfg["ft_lr"]}')
    ds  = load_dataset('nyu-mll/glue', task)['train']
    if MAX_FT_SAMPLES and len(ds) > MAX_FT_SAMPLES:
        ds = ds.shuffle(seed=SEED).select(range(MAX_FT_SAMPLES))
        print(f'  subsampled train → {len(ds)} examples (MAX_FT_SAMPLES)')
    fmt = tcfg['fmt']
    for p in model.parameters(): p.requires_grad = True
    if sum(p.numel() for p in model.parameters()) > 2e8:
        try:
            model.gradient_checkpointing_enable(); model.config.use_cache = False
            print('  gradient checkpointing enabled')
        except Exception:
            pass
    opt = torch.optim.AdamW(model.parameters(), lr=tcfg['ft_lr'])

    def _train_step(bt, bl, epoch):
        nonlocal total, nb
        enc = tok(bt, return_tensors='pt', truncation=True,
                  max_length=tcfg['max_len'], padding=True)
        opt.zero_grad()
        out  = model(input_ids=enc['input_ids'].to(DEVICE),
                     attention_mask=enc['attention_mask'].to(DEVICE))
        loss = F.cross_entropy(out.logits, torch.tensor(bl, device=DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += float(loss.item()); nb += 1
        if nb % 200 == 0:
            print(f'  epoch={epoch+1}  batch={nb}  loss={total/nb:.4f}')

    for epoch in range(done_epochs, tcfg['ft_epochs']):
        model.train()
        total, nb = 0.0, 0
        bt, bl = [], []
        for ex in ds:
            bt.append(fmt(ex)); bl.append(int(ex['label']))
            if len(bt) == tcfg['ft_bs']:
                _train_step(bt, bl, epoch); bt, bl = [], []
        if bt: _train_step(bt, bl, epoch)
        print(f'  epoch={epoch+1} done  avg_loss={total/max(nb,1):.4f}')
        if save_dir:
            model.save_pretrained(save_dir); tok.save_pretrained(save_dir)
            with open(f'{save_dir}/ft_meta.json', 'w') as f:
                json.dump({'epochs_done': epoch + 1,
                           'total_epochs': tcfg['ft_epochs']}, f)
            print(f'  [ckpt] epoch {epoch+1} → {save_dir}')
    for p in model.parameters(): p.requires_grad = False
    try: model.config.use_cache = True
    except Exception: pass
    model.eval()
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Stage B: GSN-FT + unified loader
def load_and_prepare(model_key, task):
    reg     = MODEL_REGISTRY[model_key]
    adapter = ADAPTERS[reg['family']]
    tcfg    = TASK_CONFIG[task]
    v0_key  = f'{model_key}:{task}'
    taskft_dir = f'{CKPT_DIR}/{model_key}_{task}_taskft'
    gsnft_path = f'{CKPT_DIR}/{model_key}_{task}_gsnft.pt'
    _fresh_all = (RUN_MODE == 'fresh_all')

    ckpt = reg['task_ckpt'].get(task)
    done_ep = 0
    if ckpt:
        src = ckpt
    elif os.path.isdir(taskft_dir) and not _fresh_all:
        src = taskft_dir
        meta_p = f'{taskft_dir}/ft_meta.json'
        if os.path.exists(meta_p):
            with open(meta_p) as f: done_ep = json.load(f).get('epochs_done', 0)
        else:
            done_ep = tcfg['ft_epochs']
    else:
        src = reg['hf_id']
    print(f'[load] {model_key}/{task} ← {src}'
          + (f'  (Stage-A already done {done_ep}/{tcfg["ft_epochs"]} epochs)'
             if src == taskft_dir else ''))
    tok = AutoTokenizer.from_pretrained(reg['hf_id'])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForSequenceClassification.from_pretrained(
        src, num_labels=2, dtype=torch.float32).to(DEVICE)
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tok.pad_token_id
    model.eval()

    if ckpt is None and done_ep < tcfg['ft_epochs']:
        _sa = stage_seed(model_key, task, 'stageA')
        set_all_seeds(_sa)
        print(f'  [seed] Stage-A seed={_sa}')
        task_finetune(model, tok, task, tcfg,
                      save_dir=taskft_dir, done_epochs=done_ep)
        print(f'  task-FT complete → {taskft_dir}')

    rep_mod = adapter.repair_module(model)
    print(f'  repair layer weight = {tuple(rep_mod.weight.shape)}')
    W0_orig = adapter.weight_np(model)

    head_static = adapter.build_head_static(model)
    probes = GSN_PROBES[task]
    gsn0, sn0 = compute_gsn(adapter, model, tok, probes, head_static,
                            v0_key, max_len=tcfg['max_len'])
    print(f'  GSN step=0: κ={gsn0:.4f}  ||W||₂={sn0:.4f}')
    coeff_log = [{'step': 0, 'coeff_norm': gsn0, 'spec_norm': sn0}]
    best = {'cn': gsn0, 'step': 0}

    if gsn0 >= KAPPA_TARGET:
        print(f'  κ₀={gsn0:.2f} ≥ KAPPA_TARGET={KAPPA_TARGET} → skipping GSN-FT (paper Eq.5 step-0)')
    elif os.path.exists(gsnft_path) and not _fresh_all:
        print(f'  loading cached GSN-FT state ← {gsnft_path}')
        model.load_state_dict(torch.load(gsnft_path, map_location=DEVICE))
        model.eval()
    elif FT_MAX_STEPS > 0:
        _sb = stage_seed(model_key, task, 'gsnft')
        set_all_seeds(_sb)
        print(f'  GSN-FT: {FT_MAX_STEPS} steps on {N_AUX} aux samples, lr={FT_LR}  [seed={_sb}]')
        for p in model.parameters():           p.requires_grad = False
        for p in rep_mod.parameters():         p.requires_grad = True
        for p in model.score.parameters():     p.requires_grad = True
        print(f'  trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

        aux = AUX_SAMPLES[task]
        enc = tok([t for t, _ in aux], return_tensors='pt', truncation=True,
                  max_length=tcfg['max_len'], padding=True)
        ids = enc['input_ids'].to(DEVICE); msk = enc['attention_mask'].to(DEVICE)
        lbl = torch.tensor([l for _, l in aux], dtype=torch.long, device=DEVICE)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                lr=FT_LR)
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        for step in range(1, FT_MAX_STEPS + 1):
            model.train(); opt.zero_grad()
            loss = F.cross_entropy(model(input_ids=ids, attention_mask=msk).logits, lbl)
            loss.backward(); opt.step()
            hs_step = adapter.build_head_static(model)
            cn, sn  = compute_gsn(adapter, model, tok, probes, hs_step,
                                  v0_key, max_len=tcfg['max_len'])
            coeff_log.append({'step': step, 'coeff_norm': cn, 'spec_norm': sn,
                              'loss': float(loss.item())})
            print(f'  step={step:3d}  loss={float(loss.item()):.4f}  κ={cn:.4f}  ||W||₂={sn:.4f}')
            if cn > best['cn']:
                best.update(cn=cn, step=step)
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(f'  best κ={best["cn"]:.4f} @ step={best["step"]}')
        model.load_state_dict(best_state); model = model.to(DEVICE); model.eval()
        for p in model.parameters(): p.requires_grad = False
        torch.save(model.state_dict(), gsnft_path)
        print(f'  saved GSN-FT state → {gsnft_path}')

    head_static = adapter.build_head_static(model)
    gsn_f, sn_f = compute_gsn(adapter, model, tok, probes, head_static,
                              v0_key, max_len=tcfg['max_len'])
    print(f'  GSN final: κ={gsn_f:.4f}  ||W||₂={sn_f:.4f}')
    _W0 = adapter.weight_np(model)
    print(f'  [repro] W(0) hash = {arr_hash(_W0)}   (must match across runs)')

    return {'model': model, 'tok': tok, 'adapter': adapter,
            'W0': _W0, 'W0_orig': W0_orig, 'W0_hash': arr_hash(_W0),
            'head_static': head_static, 'coeff_log': coeff_log,
            'best_step': best['step'], 'gsn_pre': gsn0, 'gsn_final': gsn_f,
            'model_key': model_key, 'task': task, 'v0_key': v0_key}

print('Stage A/B pipeline defined.')

## §5. Cache builder

In [ ]:
# Cache builder
def _np_pred(v, ctx, W):
    return 1 if forward_gap_np(W, v, ctx) > 0 else 0

def build_cache(loaded):
    model, tok, adapter = loaded['model'], loaded['tok'], loaded['adapter']
    W0, head_static     = loaded['W0'], loaded['head_static']
    task = loaded['task']; tcfg = TASK_CONFIG[task]
    fmt  = tcfg['fmt']; key = tcfg['advglue_key']; max_len = tcfg['max_len']
    rng_ = np.random.default_rng(SEED)

    def infer(text):
        return adapter.build_sample_ctx(model, tok, text, head_static, max_len)

    def cache_items(items, desc=''):
        out = []
        for i, it in enumerate(items):
            v, ctx, _ = infer(it['text'])
            p = _np_pred(v, ctx, W0)
            out.append({'text': it['text'], 'v': v, 'ctx': ctx,
                        'gt': int(it['label']), 'pred': p,
                        'correct': p == int(it['label']),
                        'method': it.get('method', 'clean')})
            if (i + 1) % 100 == 0: print(f'  {desc} {i+1}/{len(items)}')
        print(f'  {desc} done ({len(out)})')
        return out

    with open(DEV_PATH)  as f: dev_ann  = json.load(f)
    with open(TEST_PATH) as f: test_ann = json.load(f)
    dev_items = [{'text': fmt(x), 'label': x['label'],
                  'method': x.get('method', '')} for x in dev_ann[key]]
    print('Caching AdvGLUE dev...')
    dev_cache  = cache_items(dev_items, 'dev')
    repair_set = [s for s in dev_cache if not s['correct']]
    dev_acc    = 100 * sum(s['correct'] for s in dev_cache) / len(dev_cache)
    print(f'  repair |S|={len(repair_set)}  (dev acc={dev_acc:.1f}%)')

    test_items = [x for x in test_ann[key]
                  if x.get('method') not in EXCLUDED_METHODS
                  and x.get('method') != 'glue']
    by_m = defaultdict(list)
    for x in test_items:
        by_m[x['method']].append({'text': fmt(x), 'label': x['label'],
                                  'method': x['method']})
    print(f'Caching AdvGLUE test (excl. {EXCLUDED_METHODS})...')
    test_caches = {m: cache_items(v, m) for m, v in by_m.items()}

    if task == 'sst2':
        src = load_dataset('fancyzhx/yelp_polarity', split='test')
        pool = [{'text': ex['text'][:256], 'label': int(ex['label'])} for ex in src]
    else:
        src = load_dataset('nyu-mll/glue', 'rte')['train']
        pool = [{'text': fmt(ex), 'label': int(ex['label'])} for ex in src]
    rng_.shuffle(pool)

    n_rem, n_cln = tcfg['n_remain'], tcfg['n_clean']
    print('Caching remain...')
    remain_all = cache_items(pool[:n_rem * 2], 'remain')
    remain_set = [s for s in remain_all if s['correct']][:n_rem]
    if len(remain_set) < n_rem:
        print(f'  Warning: remain set short: {len(remain_set)}/{n_rem}'
              f' (W(0) accuracy too low on the 2x oversample pool — check for model degradation)')

    print('Caching clean/general...')
    c0 = n_rem * 2
    clean_all = cache_items(pool[c0: c0 + int(n_cln * 2.5)], 'clean')
    clean_set = [s for s in clean_all if s['correct']][:n_cln]
    if len(clean_set) < n_cln:
        print(f'  Warning: clean set short: {len(clean_set)}/{n_cln} (same reason as above)')

    print(f'  |P|={len(remain_set)}  |Gen|={len(clean_set)}')
    return {'repair_set': repair_set, 'remain_set': remain_set,
            'test_caches': test_caches, 'clean_set': clean_set,
            'dev_acc': dev_acc}

def load_or_build_cache(loaded, force_rebuild=False):
    cp = f"{RESULT_DIR}/cache_{loaded['model_key']}_{loaded['task']}.pkl"
    gp = f"{CKPT_DIR}/{loaded['model_key']}_{loaded['task']}_gsnft.pt"
    tp = f"{CKPT_DIR}/{loaded['model_key']}_{loaded['task']}_taskft"
    dep_mtimes = [os.path.getmtime(p) for p in (gp, tp) if os.path.exists(p)]
    stale = (os.path.exists(cp) and len(dep_mtimes) > 0
             and max(dep_mtimes) > os.path.getmtime(cp))
    if os.path.exists(cp) and not force_rebuild and not stale:
        print(f'Loading cache: {cp}')
        with open(cp, 'rb') as f: return pickle.load(f)
    d = build_cache(loaded)
    with open(cp, 'wb') as f: pickle.dump(d, f)
    print(f'Saved cache: {cp}')
    return d

print('Cache builder defined.')

## §6. QP repair

In [ ]:
# QP solver
def build_basis(W_np, rank, repair_cache, v0_key):
    d_out, d_in = W_np.shape
    if rank == 'full' or rank is None:
        return np.eye(d_out), np.arange(d_in)
    k = min(rank, min(d_out, d_in) - 1)
    U, _, _ = svds(W_np, k=k, v0=_fixed_v0(v0_key, min(d_out, d_in)))
    U = U.copy()
    Vrep = np.stack([s['v'] for s in repair_cache])
    cols = np.argsort(np.var(Vrep, axis=0))[::-1][:d_in].copy()
    return U, cols

def gap_and_coeffs(sample, W, U, cols, target_label):
    v, ctx = sample['v'], sample['ctx']
    sign   = 1.0 if target_label == 1 else -1.0
    gap    = sign * forward_gap_np(W, v, ctx)
    eff    = dgap_dpre(W, v, ctx)
    Ut_eff = U.T @ eff
    v_cols = v[cols]
    coeffs = sign * np.outer(Ut_eff, v_cols)
    return gap, coeffs

def _status_ok(status):
    s = str(status).strip().lower().replace('_', ' ')
    if 'infeasible' in s or 'non convex' in s:
        return False
    return s.startswith('solved')

def solve_qp(repair_cache, remain_cache, W, U, cols, HP):
    rank, K = U.shape[1], len(cols)
    n_s, n_p = len(repair_cache), len(remain_cache)
    nx = rank * K + n_s
    nc = n_s + n_p + n_s

    A = np.zeros((nc, nx))
    l = np.empty(nc); u = np.full(nc, 1e10)
    for i, s_ in enumerate(repair_cache):
        g, co = gap_and_coeffs(s_, W, U, cols, target_label=s_['gt'])
        A[i, :rank*K] = co.ravel()
        A[i, rank*K + i] = 1.0
        l[i] = HP['MARGIN_S'] - g
    for j, s_ in enumerate(remain_cache):
        g, co = gap_and_coeffs(s_, W, U, cols, target_label=s_['pred'])
        A[n_s + j, :rank*K] = co.ravel()
        l[n_s + j] = HP['MARGIN_H'] - g
    for i in range(n_s):
        A[n_s + n_p + i, rank*K + i] = 1.0
        l[n_s + n_p + i] = 0.0

    H = sp.diags(np.concatenate([HP['REG'] * np.ones(rank*K), np.zeros(n_s)]),
                 format='csc')
    q = np.zeros(nx); q[rank*K:] = HP['LAMBDA']
    A_sp = sp.csc_matrix(A)

    algebras = ['builtin'] if FORCE_BUILTIN_OSQP else (
        ['cuda', 'builtin'] if HAS_CUDA_OSQP else ['builtin'])
    for alg in algebras:
        for mi, eps in [(8000, 1e-5), (40000, 1e-4), (120000, 5e-4)]:
            try:
                prob = osqp.OSQP(algebra=alg)
                prob.setup(H, q, A_sp, l, u, warm_starting=True, verbose=False,
                           max_iter=mi, eps_abs=eps, eps_rel=eps,
                           adaptive_rho=True, polishing=True)
                res = prob.solve()
                if _status_ok(res.info.status):
                    if (mi, eps) != (8000, 1e-5):
                        print(f'    OSQP[{alg}] ok on retry (max_iter={mi}, '
                              f'eps={eps}, status={res.info.status})')
                    return res.x[:rank * K]
                print(f'    OSQP[{alg}] status={res.info.status} '
                      f'(max_iter={mi}, eps={eps})')
                if 'infeasible' in str(res.info.status).lower():
                    break
            except Exception as e:
                print(f'    OSQP[{alg}] exception: {e}')
                break
    return None

def apply_alpha(alpha, W_dict, U, cols):
    rank, K = U.shape[1], len(cols)
    B  = alpha.reshape(rank, K)
    dW = np.zeros_like(W_dict['W'])
    dW[:, cols] = U @ B
    W_dict['W'] = W_dict['W'] + dW
    return float(np.linalg.norm(dW, 'fro'))

In [ ]:
# Algorithm 1 (paper-faithful repair loop)
def run_repair(repair_set, remain_set, W0, HP, v0_key, tag='',
               clean_mid=None, atk_mid=None, resume_from=None):
    W = W0.copy()
    log, start_it = [], 1
    if resume_from and os.path.exists(resume_from):
        with open(resume_from, 'rb') as f:
            ck = pickle.load(f)
        W = ck['W'].copy(); log = ck['log']; start_it = len(log) + 1
        print(f'[resume] {resume_from} → iter {start_it}')

    U, cols = build_basis(W, HP['RANK'], repair_set, v0_key)
    n_s = len(repair_set)
    print(f"[Algorithm 1]  rank={HP['RANK']}  γs={HP['MARGIN_S']}  "
          f"γh={HP['MARGIN_H']}  λ={HP['LAMBDA']}  ρ={HP['REG']}  "
          f"|S|={n_s}  |P|={len(remain_set)}")
    g_sat = sum(1 for s in repair_set
                if gap_w(s['v'], s['gt'], W, s['ctx']) >= HP['MARGIN_S'] - GAP_TOL)
    print(f"Before: repair={eval_acc(repair_set, W):.1f}%  "
          f"remain={eval_acc(remain_set, W, use_gt=False):.1f}%  "
          f"gap_sat={g_sat}/{n_s}  W_hash={arr_hash(W)}")

    t0 = time.time()
    best = {'g_sat': g_sat, 'iter': 0, 'W': W.copy(), 'stats': None}
    no_improve, converged = 0, False

    for it in range(start_it, MAX_ITER + 1):
        alpha = solve_qp(repair_set, remain_set, W, U, cols, HP)
        if alpha is None:
            print(f'QP infeasible at iter={it}')
            break

        rank, K = U.shape[1], len(cols)
        dW = np.zeros_like(W)
        dW[:, cols] = U @ alpha.reshape(rank, K)
        W = W + dW
        dW_norm = float(np.linalg.norm(dW, 'fro'))

        U, cols = build_basis(W, HP['RANK'], repair_set, v0_key)

        g_sat = sum(1 for s in repair_set
                    if gap_w(s['v'], s['gt'], W, s['ctx']) >= HP['MARGIN_S'] - GAP_TOL)
        r_acc = eval_acc(repair_set, W)
        p_acc = eval_acc(remain_set, W, use_gt=False)
        try:
            spec = float(svds(W, k=1, return_singular_vectors=False)[0])
        except Exception:
            spec = float(np.linalg.norm(W, ord=2))

        entry = {'iter': it, 'repair_acc': r_acc, 'remain_acc': p_acc,
                 'gap_satisfied': g_sat, 'spec_norm': spec,
                 'dW_norm': dW_norm, 'elapsed': time.time() - t0}
        if (clean_mid or atk_mid) and it % SAVE_EVERY == 0:
            if clean_mid:
                entry['clean_acc'] = eval_acc(clean_mid, W, use_gt=False)
            if atk_mid:
                am = {m: eval_acc(c, W) for m, c in atk_mid.items()}
                entry['atk_mean'] = float(np.mean(list(am.values())))
                entry['atk_detail'] = am
            print(f"    [mid] clean={entry.get('clean_acc', float('nan')):.1f}%  "
                  f"atk={entry.get('atk_mean', float('nan')):.1f}%")
        log.append(entry)
        print(f"  iter={it:3d}  repair={r_acc:.1f}%  remain={p_acc:.1f}%  "
              f"gap_sat={g_sat}/{n_s}  ||W||={spec:.2f}  "
              f"||dW||_F={dW_norm:.3f}  {entry['elapsed']/60:.1f}m")

        if g_sat > best['g_sat'] or (g_sat == best['g_sat']
                and r_acc > (best['stats'] or {}).get('repair_acc', -1)):
            best.update(g_sat=g_sat, iter=it, W=W.copy(), stats=entry)
            no_improve = 0
        else:
            no_improve += 1

        if it % SAVE_EVERY == 0 and tag:
            with open(f'{RESULT_DIR}/qp_ckpt_{tag}_iter{it:04d}.pkl', 'wb') as f:
                pickle.dump({'W': W, 'log': log}, f)

        if g_sat == n_s and p_acc >= 100.0 - 1e-9:
            converged = True
            print(f'Converged at iter={it}: gap_sat={g_sat}/{n_s} & remain=100%')
            break
        if no_improve >= QP_PATIENCE:
            print(f'Early stop: gap_sat made no progress for {QP_PATIENCE} consecutive iters')
            break

    elapsed = time.time() - t0
    if converged:
        W_out, stats_out = W, log[-1]
    else:
        W_out = best['W']
        stats_out = best['stats'] or (log[-1] if log else None)
        print(f"[fallback] not converged → using W from best gap_sat iter={best['iter']}")
    if tag:
        with open(f'{RESULT_DIR}/qp_ckpt_{tag}_FINAL.pkl', 'wb') as f:
            pickle.dump({'W': W_out, 'log': log, 'converged': converged,
                         'final_stats': stats_out}, f)
    print(f'Done {elapsed:.1f}s  ||W0||₂={np.linalg.norm(W0, 2):.2f} → '
          f'||W~||₂={np.linalg.norm(W_out, 2):.2f}  W_hash={arr_hash(W_out)}')
    return {'W_dict': {'W': W_out.copy()}, 'log': log, 'elapsed': elapsed,
            'converged': converged, 'final_stats': stats_out}

print('Algorithm 1 (paper-faithful) defined.')

## §7. Full evaluation

In [ ]:
# full_eval + three-stage evaluation
def full_eval(W_dict, repair_set, remain_set, test_caches, clean_set,
              W_ref, HP, conv_log=None, label=''):
    W = W_dict['W']; n_s = len(repair_set)
    print(f'\n{"="*65}\nEVAL  {label}\n{"="*65}')
    r_acc  = eval_acc(repair_set, W)
    rm_acc = eval_acc(remain_set, W, use_gt=False)
    cl_acc = eval_acc(clean_set,  W, use_gt=False)
    gen    = {m: eval_acc(c, W) for m, c in test_caches.items()}
    gen_mean = float(np.mean(list(gen.values()))) if gen else float('nan')
    print(f'  Repair acc:  {r_acc:.1f}%\n  Remain acc:  {rm_acc:.1f}%')
    print(f'  General acc: {cl_acc:.1f}%  (n={len(clean_set)})')
    print(f'  Atk gen:     {gen_mean:.1f}%')
    for m in sorted(gen): print(f'    {m:<22} {gen[m]:.1f}%')

    gaps = np.array([gap_w(s['v'], s['gt'], W, s['ctx']) for s in repair_set])
    n_c1 = int((gaps >= HP['MARGIN_S'] - GAP_TOL).sum())
    print(f"\n[Cert 1] gap ≥ γs={HP['MARGIN_S']}: {n_c1}/{n_s}")
    if n_s: print(f'  min={gaps.min():.4f}  mean={gaps.mean():.4f}  max={gaps.max():.4f}')

    rgaps = np.array([gap_w(s['v'], s['pred'], W, s['ctx']) for s in remain_set])
    n_c2  = int((rgaps >= HP['MARGIN_H'] - GAP_TOL).sum())
    n_keep = int((rgaps > 0).sum())
    print(f"[Cert 2] remain gap ≥ γh={HP['MARGIN_H']}: {n_c2}/{len(remain_set)}"
          f"   (labels preserved: {n_keep}/{len(remain_set)})")
    if len(remain_set): print(f'  min={rgaps.min():.4f}  mean={rgaps.mean():.4f}')

    L_loc = np.array([lipschitz_local(W, s['ctx'], s['v']) for s in repair_set])
    if n_s and repair_set[0]['ctx']['fam'] == 'gated':
        cert3_kind = 'local per-sample (gated)'
        _eps_fn = _gated_eps_star_factory(W, repair_set[0]['ctx'])
        eps   = np.array([_eps_fn(s['v'], g) for s, g in zip(repair_set, gaps)])
        L_snd = np.where(eps > 0, gaps / (2.0 * np.maximum(eps, 1e-300)), np.inf)
    elif n_s:
        cert3_kind = 'global-L'
        L_one = lipschitz_sound(W, repair_set[0]['ctx'])
        L_snd = np.full(n_s, L_one)
        eps   = np.where(gaps > 0, gaps / (2.0 * L_snd), 0.0)
    else:
        cert3_kind = 'n/a'; L_snd = np.zeros(0); eps = np.zeros(0)
    if n_s:
        print(f'[Cert 3] ({cert3_kind})  L(eff) med={np.median(L_snd):.2e}  '
              f'L(local,diag) med={np.median(L_loc):.4f}  '
              f'tightness ≈ {np.median(L_snd/np.maximum(L_loc,1e-12)):.0f}×')
        print(f'  ε*: min={eps.min():.2e}  med={np.median(eps):.2e}  '
              f'mean={eps.mean():.2e}  max={eps.max():.2e}')

    dW = float(np.linalg.norm(W - W_ref, 'fro'))
    print(f'[dW] ||W−W_ref||_F={dW:.4f}  ratio={dW/np.linalg.norm(W_ref, "fro"):.4f}')
    if conv_log:
        ci = next((e['iter'] for e in conv_log if e['gap_satisfied'] == n_s), None)
        print(f'[CONV] gap_sat fully satisfied at iter={ci}  total_iters={len(conv_log)}')

    return {'label': label, 'repair_acc': r_acc, 'remain_acc': rm_acc,
            'clean_acc': cl_acc, 'gen': gen, 'gen_mean': gen_mean,
            'gaps': gaps, 'n_cert1': n_c1, 'n_repair': n_s,
            'rem_gaps': rgaps, 'n_cert2': n_c2, 'n_label_keep': n_keep,
            'L_sound': L_snd, 'L_local': L_loc, 'eps_star': eps,
            'cert3_kind': cert3_kind, 'dW_fro': dW}

def three_stage_eval(loaded, DATA, repair_result, HP,
                     repair_set=None, n_remain_eval=200):
    rset = repair_set if repair_set is not None else DATA['repair_set']
    rem  = DATA['remain_set'][:n_remain_eval]
    e_ref = full_eval({'W': loaded['W0_orig'].copy()}, rset, rem,
                      DATA['test_caches'], DATA['clean_set'],
                      loaded['W0_orig'], HP,
                      label='W0-swap (reference only — ctx is post-GSN-FT)')
    e_bef = full_eval({'W': loaded['W0'].copy()}, rset, rem,
                      DATA['test_caches'], DATA['clean_set'],
                      loaded['W0'], HP, label='GSN-FT (QP-before)')
    e_aft = full_eval(repair_result['W_dict'], rset, rem,
                      DATA['test_caches'], DATA['clean_set'],
                      loaded['W0'], HP, conv_log=repair_result['log'],
                      label='WARP (QP-after)')
    print(f'\n{"="*70}\nSUMMARY  {loaded["model_key"]}/{loaded["task"]}  '
          f'|S|={len(rset)}  |P|={len(rem)}\n{"="*70}')
    print(f'  {"":<22}{"Repair":>9}{"Remain":>9}{"General":>9}{"AtkGen":>9}')
    for lab, e in [('W0-swap(ref)', e_ref), ('GSN-FT before', e_bef),
                   ('WARP after', e_aft)]:
        print(f'  {lab:<22}{e["repair_acc"]:>8.1f}%{e["remain_acc"]:>8.1f}%'
              f'{e["clean_acc"]:>8.1f}%{e["gen_mean"]:>8.1f}%')
    return {'ref': e_ref, 'before': e_bef, 'after': e_aft}

print('full_eval + three_stage_eval defined.')

## §8. Baselines: same-layer LoRA + all-layers Full FT

Training protocol: lr=1e-4, bs=16, max 300 epochs, early stop (patience 5, tol 1e-6).

In [ ]:
# LoRA wrapper + baseline training helpers
class LoRAWrap(nn.Module):
    def __init__(self, base_mod, r, alpha):
        super().__init__()
        self.base = base_mod
        self.is_conv1d = hasattr(base_mod, 'nf')
        if self.is_conv1d:
            d_in, d_out = base_mod.weight.shape
        else:
            d_out, d_in = base_mod.weight.shape
        self.A = nn.Parameter(torch.zeros(d_in, r))
        self.B = nn.Parameter(torch.zeros(r, d_out))
        nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)
        self.scaling = alpha / r
        for p in self.base.parameters(): p.requires_grad = False

    def forward(self, x):
        return self.base(x) + (x @ self.A) @ self.B * self.scaling

    def merged_delta_np(self):
        with torch.no_grad():
            d = (self.A @ self.B) * self.scaling
        return d.detach().cpu().numpy().astype(np.float64).T.copy()

def baseline_train(model, tok, texts, labels, name, max_len):
    enc = tok(texts, return_tensors='pt', truncation=True,
              max_length=max_len, padding=True)
    ids = enc['input_ids'].to(DEVICE); msk = enc['attention_mask'].to(DEVICE)
    lbl = torch.tensor(labels, dtype=torch.long, device=DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    print(f'  [{name}] trainable={sum(p.numel() for p in params):,}  n_train={len(texts)}')
    opt = torch.optim.AdamW(params, lr=BASELINE_LR)
    best, wait, n = float('inf'), 0, ids.size(0)
    for ep in range(BASELINE_MAX_EPOCHS):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        tot, nb = 0.0, 0
        for i in range(0, n, BASELINE_BS):
            idx = perm[i:i + BASELINE_BS]
            opt.zero_grad()
            loss = F.cross_entropy(
                model(input_ids=ids[idx], attention_mask=msk[idx]).logits, lbl[idx])
            loss.backward(); opt.step()
            tot += float(loss.item()); nb += 1
        avg = tot / max(nb, 1)
        if ep % 20 == 0 or ep < 3: print(f'  [{name}] epoch={ep}  loss={avg:.6f}')
        if best - avg < BASELINE_TOL:
            wait += 1
            if wait >= BASELINE_PATIENCE:
                print(f'  [{name}] early stop @ epoch={ep}  loss={avg:.6f}'); break
        else:
            wait = 0
        best = min(best, avg)
    model.eval()

def recache_and_eval_ftmodel(model, adapter, tok, DATA, HP, max_len, label,
                             repair_set=None, n_remain=200):
    hs  = adapter.build_head_static(model)
    W_n = adapter.weight_np(model)
    def pred_of(text):
        v, ctx, _ = adapter.build_sample_ctx(model, tok, text, hs, max_len)
        return _np_pred(v, ctx, W_n)
    def acc(cache, ref_key, desc):
        ok = 0
        for i, s in enumerate(cache):
            ok += (pred_of(s['text']) == s[ref_key])
            if (i + 1) % 200 == 0: print(f'    {desc} {i+1}/{len(cache)}')
        return 100.0 * ok / max(len(cache), 1)
    rset = repair_set if repair_set is not None else DATA['repair_set']
    print(f'  [{label}] re-encoding fixed sets with FT model...')
    r_acc  = acc(rset, 'gt',   'repair')
    rm_acc = acc(DATA['remain_set'][:n_remain], 'pred', 'remain')
    cl_acc = acc(DATA['clean_set'],  'gt',   'clean')
    gen    = {m: acc(c, 'gt', m) for m, c in DATA['test_caches'].items()}
    gm     = float(np.mean(list(gen.values()))) if gen else float('nan')
    print(f'  [{label}] repair={r_acc:.1f}%  remain={rm_acc:.1f}%  '
          f'clean={cl_acc:.1f}%  atkgen={gm:.1f}%')
    return {'label': label, 'repair_acc': r_acc, 'remain_acc': rm_acc,
            'clean_acc': cl_acc, 'gen': gen, 'gen_mean': gm}

In [ ]:
# run_baselines: LoRA + Full FT
def run_baselines(loaded, DATA, HP, repair_set=None,
                  n_remain_train=None, train_head=False):
    tok, adapter = loaded['tok'], loaded['adapter']
    tcfg = TASK_CONFIG[loaded['task']]; max_len = tcfg['max_len']
    rset = repair_set if repair_set is not None else DATA['repair_set']
    n_rt = n_remain_train if n_remain_train is not None else N_REMAIN_QP

    d_out, d_in = loaded['W0'].shape
    r_lora = (max(1, round(HP['RANK'] * d_in / (d_in + d_out)))
              if LORA_MATCH_DOF else HP['RANK'])
    print(f'[DOF] WARP decision vars = {HP["RANK"]}×{d_in} = {HP["RANK"]*d_in:,}   '
          f'LoRA r={r_lora} params = {r_lora}×({d_in}+{d_out}) = {r_lora*(d_in+d_out):,}')

    tr_txt = [s['text'] for s in rset] + \
             [s['text'] for s in DATA['remain_set'][:n_rt]]
    tr_lbl = [s['gt'] for s in rset] + \
             [s['gt'] for s in DATA['remain_set'][:n_rt]]

    loaded['model'].to('cpu'); gc.collect(); torch.cuda.empty_cache()

    lora_runs, full_runs = [], []
    for _si in range(BASELINE_N_SEEDS):
        _seed = stage_seed(loaded['model_key'], loaded['task'], f'baseline{_si}')
        set_all_seeds(_seed)
        if BASELINE_N_SEEDS > 1:
            print(f"\n────── baseline seed {_si+1}/{BASELINE_N_SEEDS} (seed={_seed}) ──────")

        print(f'\n{"="*70}\nBASELINE 1: Same-layer LoRA (r={r_lora})\n{"="*70}')
        m_lora = copy.deepcopy(loaded['model']).to(DEVICE)
        for p in m_lora.parameters(): p.requires_grad = False
        base_mod = adapter.repair_module(m_lora)
        lora = LoRAWrap(base_mod, r=r_lora, alpha=r_lora * 2).to(DEVICE)
        adapter.set_repair_module(m_lora, lora)
        if train_head:
            for p in m_lora.score.parameters(): p.requires_grad = True
        baseline_train(m_lora, tok, tr_txt, tr_lbl, 'LoRA', max_len)

        W_merged = loaded['W0'] + lora.merged_delta_np()
        rs_e, rem_e = rset, DATA['remain_set'][:n_rt]
        cl_e, tc_e  = DATA['clean_set'], DATA['test_caches']
        if train_head:
            Wsc = m_lora.score.weight.detach().cpu().numpy().astype(np.float64)
            wd  = (Wsc[1] - Wsc[0]).copy()
            _patch = lambda cache: [{**s, 'ctx': {**s['ctx'], 'w_diff': wd}}
                                    for s in cache]
            rs_e, rem_e = _patch(rs_e), _patch(rem_e)
            cl_e = _patch(cl_e)
            tc_e = {m: _patch(c) for m, c in tc_e.items()}
        e_lora = full_eval({'W': W_merged}, rs_e, rem_e, tc_e, cl_e,
                           loaded['W0'], HP,
                           label=f'LoRA r={r_lora}' + (' (+head)' if train_head else ''))
        lora_runs.append(e_lora)
        del m_lora; gc.collect(); torch.cuda.empty_cache()

        print(f'\n{"="*70}\nBASELINE 2: All-layers Full FT\n{"="*70}')
        m_full = copy.deepcopy(loaded['model']).to(DEVICE)
        for p in m_full.parameters(): p.requires_grad = True
        if sum(p.numel() for p in m_full.parameters()) > 2e8:
            try:
                m_full.gradient_checkpointing_enable()
                m_full.config.use_cache = False
                print('  [FullFT] gradient checkpointing enabled')
            except Exception:
                pass
        baseline_train(m_full, tok, tr_txt, tr_lbl, 'FullFT', max_len)
        full_runs.append(recache_and_eval_ftmodel(
            m_full, adapter, tok, DATA, HP, max_len, 'Full FT',
            repair_set=rset, n_remain=n_rt))
        del m_full; gc.collect(); torch.cuda.empty_cache()

    def _agg(runs):
        base = dict(runs[0])
        for k in ('repair_acc', 'remain_acc', 'clean_acc', 'gen_mean'):
            vals = [r[k] for r in runs]
            base[k] = float(np.mean(vals)); base[k + '_std'] = float(np.std(vals))
        base['gen'] = {m: float(np.mean([r['gen'][m] for r in runs]))
                       for m in runs[0]['gen']}
        base['per_seed'] = [{k: r[k] for k in
                            ('repair_acc', 'remain_acc', 'clean_acc', 'gen_mean')}
                           for r in runs]
        base['n_seeds'] = len(runs)
        return base
    results = {'lora': _agg(lora_runs), 'full_ft': _agg(full_runs)}
    if BASELINE_N_SEEDS > 1:
        for name in ('lora', 'full_ft'):
            e = results[name]
            print(f"  [{name}] over {e['n_seeds']} seeds: "
                  f"repair={e['repair_acc']:.1f}±{e['repair_acc_std']:.1f}  "
                  f"clean={e['clean_acc']:.1f}±{e['clean_acc_std']:.1f}  "
                  f"atkgen={e['gen_mean']:.1f}±{e['gen_mean_std']:.1f}")
    return results

print('Baselines defined.')

## §9. Ablations (matches paper's Ablation Studies)

Four sweeps, all built on `run_repair`, each varying one factor; evaluated on a mid-size subset (clean 500 / full per-attack) to save time. Recommended: only set `RUN_ABLATIONS=True` for the winner model.

In [ ]:
# Ablation helpers
def _quick_eval(W, DATA, HP, n_clean=500):
    return {
        'repair_acc': eval_acc(DATA['repair_set'], W),
        'remain_acc': eval_acc(DATA['remain_set'][:100], W, use_gt=False),
        'clean_acc':  eval_acc(DATA['clean_set'][:n_clean], W, use_gt=False),
        'gen_mean':   float(np.mean([eval_acc(c, W)
                                     for c in DATA['test_caches'].values()])),
    }

def ablate_rank(loaded, DATA, HP, ranks=(2, 8, 16, 32, 64)):
    rows = []
    for r in ranks:
        hp = {**HP, 'RANK': r}
        print(f'\n===== ABLATION rank={r} =====')
        res = run_repair(DATA['repair_set'], DATA['remain_set'][:100],
                         loaded['W0'], hp, loaded['v0_key'],
                         tag=f'{loaded["model_key"]}_{loaded["task"]}_abl_rank{r}')
        rows.append({'rank': r, 'converged': res['converged'],
                     'iters': len(res['log']), **_quick_eval(res['W_dict']['W'], DATA, hp)})
    _print_rows(rows, 'rank')
    return rows

def ablate_repair_size(loaded, DATA, HP, sizes=(5, 10, 20, 40, None)):
    init_gaps = np.array([gap_w(s['v'], s['gt'], loaded['W0'], s['ctx'])
                          for s in DATA['repair_set']])
    order = np.argsort(-init_gaps)
    rows = []
    for n in sizes:
        sub = [DATA['repair_set'][i] for i in (order if n is None else order[:n])]
        hp = dict(HP)
        print(f'\n===== ABLATION |S|={len(sub)} =====')
        res = run_repair(sub, DATA['remain_set'][:100], loaded['W0'], hp,
                         loaded['v0_key'],
                         tag=f'{loaded["model_key"]}_{loaded["task"]}_abl_S{len(sub)}')
        row = {'n_repair': len(sub), 'converged': res['converged'],
               'iters': len(res['log']), 'elapsed_s': res['elapsed'],
               **_quick_eval(res['W_dict']['W'], DATA, hp)}
        rows.append(row)
    _print_rows(rows, 'n_repair')
    return rows

def ablate_remain_size(loaded, DATA, HP, sizes=(0, 50, 100, 200)):
    rows = []
    for n in sizes:
        rem = DATA['remain_set'][:n]
        print(f'\n===== ABLATION |P|={n} =====')
        res = run_repair(DATA['repair_set'], rem, loaded['W0'], HP,
                         loaded['v0_key'],
                         tag=f'{loaded["model_key"]}_{loaded["task"]}_abl_P{n}')
        rows.append({'n_remain': n, 'converged': res['converged'],
                     'iters': len(res['log']),
                     **_quick_eval(res['W_dict']['W'], DATA, HP)})
    _print_rows(rows, 'n_remain')
    return rows

def ablate_hyper_grid(loaded, DATA, HP,
                      regs=(2.0, 5.0), lambdas=(10.0, 50.0), margins=(0.1, 0.5, 1.0)):
    rows = []
    for reg in regs:
        for lam in lambdas:
            for ms in margins:
                hp = {**HP, 'REG': reg, 'LAMBDA': lam, 'MARGIN_S': ms}
                print(f'\n===== GRID ρ={reg} λ={lam} γs={ms} =====')
                res = run_repair(DATA['repair_set'], DATA['remain_set'][:100],
                                 loaded['W0'], hp, loaded['v0_key'], tag='')
                rows.append({'REG': reg, 'LAMBDA': lam, 'MARGIN_S': ms,
                             'converged': res['converged'], 'iters': len(res['log']),
                             **_quick_eval(res['W_dict']['W'], DATA, hp)})
    _print_rows(rows, 'grid')
    return rows

def _print_rows(rows, name):
    if not rows: return
    keys = list(rows[0].keys())
    print(f'\n─── ABLATION [{name}] ───')
    print('  ' + '  '.join(f'{k:>12}' for k in keys))
    for r in rows:
        print('  ' + '  '.join(
            f'{r[k]:>11.1f}%' if isinstance(r[k], float) and k.endswith('acc') or k == 'gen_mean'
            else f'{str(r[k]):>12}' for k in keys))

print('Ablations defined.')

## §10. Orchestrator: single-group pipeline + full matrix + winner selection

`run_one(model_key, task)` runs one group end to end: load (Stage A/B) → cache → QP repair → three-stage eval → baselines → (ablations). Each result is saved to `{RESULT_DIR}/result_{model_key}_{task}.pkl`; skipped if it already exists.

`print_summary()` reads the result pkls and prints the summary table, marking **winner = WARP's AtkGen beats both LoRA and Full FT**.

In [ ]:
# curate_repair_set + run_one (WARP only) + add_baselines
def curate_repair_set(W0, DATA):
    rem_gaps0 = np.array([gap_w(s['v'], s['pred'], W0, s['ctx'])
                          for s in DATA['remain_set']])
    scale = float(np.median(rem_gaps0))
    if REPAIR_GAP_FLOOR_MODE == 'relative':
        floor = -REPAIR_GAP_FLOOR_REL * scale
    elif REPAIR_GAP_FLOOR_MODE == 'absolute':
        floor = REPAIR_GAP_FLOOR_ABS
    else:
        floor = None
    init_gaps = np.array([gap_w(s['v'], s['gt'], W0, s['ctx'])
                          for s in DATA['repair_set']])
    if floor is not None:
        keep = init_gaps >= floor
        repair_used = [s for s, k in zip(DATA['repair_set'], keep) if k]
        dropped = init_gaps[~keep]
    else:
        repair_used, dropped = DATA['repair_set'], np.array([])
    return repair_used, floor, scale, dropped

def _settings_hash(HP, floor):
    return hashlib.md5(json.dumps(
        {**HP, 'floor': None if floor is None else round(floor, 2),
         'n_rem_qp': N_REMAIN_QP, 'algo': ALGO_TAG},
        sort_keys=True).encode()).hexdigest()[:6]

def _run_cfg():
    return {'N_REMAIN_QP': N_REMAIN_QP,
            'floor_mode': REPAIR_GAP_FLOOR_MODE,
            'floor_rel':  REPAIR_GAP_FLOOR_REL,
            'floor_abs':  REPAIR_GAP_FLOOR_ABS,
            'algo': ALGO_TAG}

def run_one(model_key, task, force=False):
    reg_ = MODEL_REGISTRY[model_key]
    HP = {**reg_['qp'], **reg_.get('qp_task', {}).get(task, {})}
    cur_cfg = _run_cfg()
    fresh_qp = RUN_MODE in ('fresh_qp', 'fresh_all')
    out_path = f'{RESULT_DIR}/result_{model_key}_{task}.pkl'

    if os.path.exists(out_path) and not force and not fresh_qp:
        with open(out_path, 'rb') as f: prev = pickle.load(f)
        if prev.get('HP') == HP and prev.get('run_cfg') == cur_cfg:
            print(f'[skip] {out_path} already exists with matching HP/run_cfg')
            return prev
        print(f'[stale] {out_path} settings differ from current config → rerunning')
    print(f'\n{"#"*78}\n#  WARP  {model_key} × {task}   HP={HP}   '
          f'ALGO={ALGO_TAG}\n{"#"*78}')

    loaded = load_and_prepare(model_key, task)
    DATA   = load_or_build_cache(loaded, force_rebuild=(RUN_MODE == 'fresh_all'))
    print(f"[sanity] pre-QP: repair={eval_acc(DATA['repair_set'], loaded['W0']):.1f}% "
          f"(expect 0)  remain={eval_acc(DATA['remain_set'], loaded['W0'], use_gt=False):.1f}% "
          f"(expect 100)")

    repair_used, floor, scale, dropped = curate_repair_set(loaded['W0'], DATA)
    if floor is not None:
        print(f'[curation] gap scale median(remain gap)={scale:.2f} → '
              f'floor={floor:.2f}: kept {len(repair_used)}/{len(DATA["repair_set"])}'
              + (f' (dropped gaps: {np.round(np.sort(dropped), 2).tolist()})'
                 if len(dropped) else ''))
    remain_qp = DATA['remain_set'][:N_REMAIN_QP]

    tag = f'{model_key}_{task}_{_settings_hash(HP, floor)}'
    print(f'[tag] {tag}')
    cache_path = f'{RESULT_DIR}/cache_{model_key}_{task}.pkl'
    final_ck   = f'{RESULT_DIR}/qp_ckpt_{tag}_FINAL.pkl'

    def _fresh(p):
        return os.path.exists(p) and (not os.path.exists(cache_path)
               or os.path.getmtime(p) >= os.path.getmtime(cache_path))

    if (not fresh_qp) and (not force) and _fresh(final_ck):
        print(f'[resume] QP already complete → loading {final_ck}')
        with open(final_ck, 'rb') as f: ck = pickle.load(f)
        rep = {'W_dict': {'W': ck['W'].copy()}, 'log': ck['log'],
               'elapsed': ck['log'][-1]['elapsed'] if ck['log'] else 0.0,
               'converged': ck.get('converged', False),
               'final_stats': ck.get('final_stats')}
    else:
        resume = None
        if (not fresh_qp) and (not force):
            import glob as _glob
            iters = sorted(p for p in _glob.glob(
                f'{RESULT_DIR}/qp_ckpt_{tag}_iter*.pkl') if _fresh(p))
            resume = iters[-1] if iters else None
        atk_mid = {m: c[:50] for m, c in DATA['test_caches'].items()}
        rep = run_repair(repair_used, remain_qp, loaded['W0'], HP,
                         loaded['v0_key'], tag=tag,
                         clean_mid=DATA['clean_set'][:200], atk_mid=atk_mid,
                         resume_from=resume)

    evals = three_stage_eval(loaded, DATA, rep, HP,
                            repair_set=repair_used, n_remain_eval=N_REMAIN_QP)

    result = {
        'model_key': model_key, 'task': task, 'HP': HP,
        'run_cfg': cur_cfg, 'run_mode': RUN_MODE, 'algo': ALGO_TAG,
        'tag': tag, 'env': ENV_FINGERPRINT,
        'repro': {'W0_hash': loaded.get('W0_hash'),
                  'W_repaired_hash': arr_hash(rep['W_dict']['W']),
                  'seed_stageA': stage_seed(model_key, task, 'stageA'),
                  'seed_gsnft':  stage_seed(model_key, task, 'gsnft')},
        'curation': {'floor': floor, 'mode': REPAIR_GAP_FLOOR_MODE,
                     'gap_scale': scale, 'n_total': len(DATA['repair_set']),
                     'n_used': len(repair_used),
                     'dropped_init_gaps': np.sort(dropped).tolist()},
        'converged': rep['converged'], 'qp_log': rep['log'],
        'elapsed': rep['elapsed'],
        'gsn': {'pre': loaded['gsn_pre'], 'post': loaded['gsn_final'],
                'best_step': loaded['best_step'], 'coeff_log': loaded['coeff_log']},
        'evals': evals,
        'baselines': {},
        'ablations': {},
        'W_repaired': rep['W_dict']['W'],
        'set_sizes': {'S': len(repair_used),
                      'S_precuration': len(DATA['repair_set']),
                      'P': len(remain_qp), 'Gen': len(DATA['clean_set']),
                      'AtkGen': sum(len(c) for c in DATA['test_caches'].values())},
    }
    with open(out_path, 'wb') as f: pickle.dump(result, f)
    print(f'[saved] {out_path}')
    print(f"[repro] W(0)={result['repro']['W0_hash']}  "
          f"W~={result['repro']['W_repaired_hash']}  "
          f"converged={rep['converged']}")

    del loaded['model']; del loaded
    gc.collect(); torch.cuda.empty_cache()
    return result

def add_baselines(model_key, task, force=False):
    out_path = f'{RESULT_DIR}/result_{model_key}_{task}.pkl'
    if not os.path.exists(out_path):
        print(f'[skip] {model_key}/{task}: WARP result missing, run run_warp_matrix() first')
        return None
    with open(out_path, 'rb') as f: R = pickle.load(f)
    if R.get('baselines') and not force:
        print(f'[skip] {model_key}/{task}: baseline already exists (force=True to rerun)')
        return R

    print(f'\n{"#"*78}\n#  BASELINES  {model_key} × {task}   HP={R["HP"]}\n{"#"*78}')
    loaded = load_and_prepare(model_key, task)
    DATA   = load_or_build_cache(loaded)

    w0h = arr_hash(loaded['W0'])
    if w0h != R['repro']['W0_hash']:
        print(f'Warning: W(0) hash mismatch! WARP run={R["repro"]["W0_hash"]} '
              f'now={w0h} — baseline and WARP do not share the same starting point, results not comparable.')
        print('  Possible cause: Stage-A/GSN-FT checkpoint changed, or environment settings changed.')
    else:
        print(f'[check] W(0) hash matches: {w0h}')

    repair_used, floor, scale, _ = curate_repair_set(loaded['W0'], DATA)
    if len(repair_used) != R['set_sizes']['S']:
        print(f'Warning: curated |S| mismatch! WARP run={R["set_sizes"]["S"]} '
              f'now={len(repair_used)}')
    else:
        print(f'[check] curated |S| matches: {len(repair_used)}')

    base = run_baselines(loaded, DATA, R['HP'], repair_set=repair_used,
                         n_remain_train=N_REMAIN_QP)
    R['baselines'] = base
    with open(out_path, 'wb') as f: pickle.dump(R, f)
    print(f'[saved] {out_path} (baselines added)')

    del loaded['model']; del loaded
    gc.collect(); torch.cuda.empty_cache()
    return R

In [ ]:
# run_warp_matrix / run_baseline_matrix / print_summary
def _baseline_valid(e):
    return bool(e) and e['remain_acc'] >= 90.0 and e['repair_acc'] > 0.0

def _safe_loop(fn, force_keys=()):
    out = {}
    for mk, tk in RUN_MATRIX:
        fc = (force_keys is True) or ((mk, tk) in force_keys)
        try:
            r = fn(mk, tk, force=fc)
            if r is not None:
                out[(mk, tk)] = r
        except Exception as e:
            msg = f'{type(e).__name__}: {e}'
            import traceback as _tb
            _tb.clear_frames(e.__traceback__)
            del e
            gc.collect(); torch.cuda.empty_cache()
            print(f'[ERROR] {mk}/{tk}: {msg}')
    return out

def run_warp_matrix(force_keys=()):
    print(f'\n{"="*78}\n  Stage 1: WARP × {len(RUN_MATRIX)} groups  (ALGO={ALGO_TAG})\n{"="*78}')
    res = _safe_loop(run_one, force_keys)
    print(f'\n{"="*78}\n  Stage 1 done: {len(res)}/{len(RUN_MATRIX)} groups\n{"="*78}')
    for (mk, tk), r in res.items():
        w = r['evals']['after']
        print(f'  {mk+"/"+tk:<24} converged={str(r["converged"]):<6} '
              f'repair={w["repair_acc"]:6.1f}%  remain={w["remain_acc"]:6.1f}%  '
              f'W~={r["repro"]["W_repaired_hash"]}')
    bad = [f'{mk}/{tk}' for (mk, tk), r in res.items() if not r['converged']]
    print(f'\nNot converged: {", ".join(bad)}' if bad
          else '\nAll converged (repair=100% and remain=100%, Theorem 3.1 premise holds)')
    return res

def run_baseline_matrix(force_keys=()):
    print(f'\n{"="*78}\n  Stage 2: baselines × {len(RUN_MATRIX)} groups\n{"="*78}')
    res = _safe_loop(add_baselines, force_keys)
    print(f'\n{"="*78}\n  Stage 2 done: {len(res)}/{len(RUN_MATRIX)} groups\n{"="*78}')
    return res

def load_all_results():
    out = {}
    for mk, tk in RUN_MATRIX:
        p = f'{RESULT_DIR}/result_{mk}_{tk}.pkl'
        if os.path.exists(p):
            with open(p, 'rb') as f: out[(mk, tk)] = pickle.load(f)
    return out

def print_summary(save_md=True):
    all_res = load_all_results()
    if not all_res:
        print('No result pkl found, run run_warp_matrix() first'); return {}, []
    has_base = any(r.get('baselines') for r in all_res.values())
    print(f'\n{"="*100}')
    print(f'{"model/task":<24}{"method":<16}{"Repair":>9}{"Remain":>9}'
          f'{"General":>9}{"AtkGen":>9}   win?')
    print('=' * 100)
    winners, near_misses = [], []
    md = ['| Model | Task | Method | Repair | Remain | General | AtkGen | Mark |',
          '|---|---|---|---|---|---|---|---|']
    for (mk, tk), r in all_res.items():
        w  = r['evals']['after']
        b  = r.get('baselines') or {}
        lo, fu = b.get('lora'), b.get('full_ft')
        pool = [e['gen_mean'] for e in (lo, fu) if _baseline_valid(e)]
        base_best     = max(pool) if pool else None
        repaired_full = w['repair_acc'] >= 100.0 - 1e-9
        beats         = (base_best is not None) and (w['gen_mean'] > base_best)
        is_win = bool(lo and fu and repaired_full and beats)
        near   = bool(lo and fu and beats and not repaired_full)
        if is_win: winners.append((mk, tk))
        if near:   near_misses.append((mk, tk))
        rows = [('WARP (ours)', w)] + \
               ([('LoRA', lo)] if lo else []) + ([('Full FT', fu)] if fu else [])
        for name, e in rows:
            if name.startswith('WARP'):
                mark = '★' if is_win else ('≈' if near else
                                           ('' if has_base else '(no baseline)'))
            else:
                mark = '†' if not _baseline_valid(e) else ''
            print(f'{mk+"/"+tk:<24}{name:<16}{e["repair_acc"]:>8.1f}%'
                  f'{e["remain_acc"]:>8.1f}%{e["clean_acc"]:>8.1f}%'
                  f'{e["gen_mean"]:>8.1f}% {mark}')
            md.append(f'| {mk} | {tk} | {name} | {e["repair_acc"]:.1f}% | '
                      f'{e["remain_acc"]:.1f}% | {e["clean_acc"]:.1f}% | '
                      f'{e["gen_mean"]:.1f}% | {mark} |')
        print('-' * 100)

    legend = ('★ winner (repair 100% and AtkGen > all valid baselines)   '
              '≈ near-miss (beats AtkGen on everything but repair not full)   '
              '† baseline training failed (remain<90% or repair=0%), excluded from winner check')
    print(f'\n{legend}')
    if not has_base:
        print('\n(baselines not run yet — run run_baseline_matrix() to see winner check)')
    else:
        print('\n★ WINNERS:')
        for mk, tk in winners: print(f'   {mk} × {tk}')
        if near_misses:
            print('≈ NEAR-MISS:')
            for mk, tk in near_misses: print(f'   {mk} × {tk}')

    print(f'\n{"="*100}\nPER-ATTACK BREAKDOWN (WARP after)\n{"="*100}')
    methods = sorted({m for r in all_res.values() for m in r['evals']['after']['gen']})
    print(f'{"model/task":<24}' + ''.join(f'{m:>13}' for m in methods) + f'{"mean":>9}')
    md += ['', '### Per-attack breakdown (WARP after)', '',
           '| Model | Task | ' + ' | '.join(methods) + ' | Mean |',
           '|---' * (len(methods) + 3) + '|']
    for (mk, tk), r in all_res.items():
        g = r['evals']['after']['gen']
        print(f'{mk+"/"+tk:<24}' + ''.join(f'{g.get(m, float("nan")):>12.1f}%'
                                           for m in methods)
              + f'{r["evals"]["after"]["gen_mean"]:>8.1f}%')
        md.append(f'| {mk} | {tk} | '
                  + ' | '.join(f'{g.get(m, float("nan")):.1f}%' for m in methods)
                  + f' | {r["evals"]["after"]["gen_mean"]:.1f}% |')

    print(f'\n{"="*100}\nREPRODUCIBILITY / CONVERGENCE\n{"="*100}')
    print(f'{"model/task":<24}{"conv":>7}{"iters":>7}{"|S|":>6}{"|P|":>6}  '
          f'{"W(0) hash":<14}{"W~ hash":<14}{"HP"}')
    md += ['', '### Reproducibility', '',
           '| Model | Task | converged | iters | \\|S\\| | W(0) hash | W~ hash | HP |',
           '|---|---|---|---|---|---|---|---|']
    for (mk, tk), r in all_res.items():
        rp = r.get('repro', {})
        print(f'{mk+"/"+tk:<24}{str(r.get("converged")):>7}'
              f'{len(r.get("qp_log", [])):>7}{r["set_sizes"]["S"]:>6}'
              f'{r["set_sizes"]["P"]:>6}  {str(rp.get("W0_hash")):<14}'
              f'{str(rp.get("W_repaired_hash")):<14}{r["HP"]}')
        md.append(f'| {mk} | {tk} | {r.get("converged")} | '
                  f'{len(r.get("qp_log", []))} | {r["set_sizes"]["S"]} | '
                  f'{rp.get("W0_hash")} | {rp.get("W_repaired_hash")} | {r["HP"]} |')
    bad = [f'{mk}/{tk}' for (mk, tk), r in all_res.items() if not r.get('converged')]
    print(f'\nNot converged: {", ".join(bad)}' if bad else '\nAll converged')

    if save_md:
        with open(f'{RESULT_DIR}/summary_matrix.md', 'w') as f:
            f.write('\n'.join(md) + '\n\n' + legend + '\n')
        print(f'\n[saved] {RESULT_DIR}/summary_matrix.md')
    return all_res, winners

In [ ]:
# sweep_hp / sweep_margin
def sweep_hp(model_key, task, hp_name, values):
    reg_ = MODEL_REGISTRY[model_key]
    HP0 = {**reg_['qp'], **reg_.get('qp_task', {}).get(task, {})}
    loaded = load_and_prepare(model_key, task)
    DATA   = load_or_build_cache(loaded)

    repair_used, floor, scale, _ = curate_repair_set(loaded['W0'], DATA)
    remain_qp = DATA['remain_set'][:N_REMAIN_QP]
    print(f'[sweep] {model_key}/{task}  |S|={len(repair_used)}  '
          f'floor={floor:.2f}  {hp_name}={list(values)}')

    rows = []
    for ms in values:
        hp = {**HP0, hp_name: (int(ms) if hp_name == 'RANK' else float(ms))}
        import hashlib
        sh = hashlib.md5(json.dumps(
            {**hp, 'floor': None if floor is None else round(floor, 2),
             'n_rem_qp': N_REMAIN_QP}, sort_keys=True).encode()).hexdigest()[:6]
        print(f'\n===== {hp_name}={ms}  (tag …_{sh}) =====')
        res = run_repair(repair_used, remain_qp, loaded['W0'], hp,
                         loaded['v0_key'], tag=f'{model_key}_{task}_{sh}')
        W = res['W_dict']['W']
        rows.append({hp_name: ms, 'converged': res['converged'],
                     'iters': len(res['log']),
                     'repair_acc': eval_acc(repair_used, W),
                     'remain_acc': eval_acc(remain_qp, W, use_gt=False),
                     'clean_acc':  eval_acc(DATA['clean_set'], W, use_gt=False),
                     'gen_mean':   float(np.mean([eval_acc(c, W)
                                    for c in DATA['test_caches'].values()])),
                     'dW_ratio':   float(np.linalg.norm(W - loaded['W0'], 'fro')
                                   / np.linalg.norm(loaded['W0'], 'fro'))})
        print(f"  → repair={rows[-1]['repair_acc']:.1f}%  "
              f"clean={rows[-1]['clean_acc']:.1f}%  "
              f"atkgen={rows[-1]['gen_mean']:.1f}%  dW={rows[-1]['dW_ratio']:.3f}")

    print(f'\n─── {hp_name} sweep  {model_key}/{task} ───')
    print(f'  {hp_name:>8}{"conv":>6}{"iters":>7}{"Repair":>9}{"Remain":>9}'
          f'{"General":>9}{"AtkGen":>9}{"dW":>8}')
    for r in rows:
        print(f'  {r[hp_name]:>8}{str(r["converged"]):>6}{r["iters"]:>7}'
              f'{r["repair_acc"]:>8.1f}%{r["remain_acc"]:>8.1f}%'
              f'{r["clean_acc"]:>8.1f}%{r["gen_mean"]:>8.1f}%{r["dW_ratio"]:>8.3f}')
    with open(f'{RESULT_DIR}/sweep_{hp_name}_{model_key}_{task}.pkl', 'wb') as f:
        pickle.dump(rows, f)
    del loaded['model']; del loaded
    gc.collect(); torch.cuda.empty_cache()
    return rows

def sweep_margin(model_key, task, margins):
    return sweep_hp(model_key, task, 'MARGIN_S', margins)

print('sweep_hp / sweep_margin ready.')

In [ ]:
# Version check
PATCH_VERSION = '2026-07-25-v11-paper-faithful'
import inspect as _insp

assert 'ALGO_TAG' in globals() and ALGO_TAG == 'paper-faithful-alg1', \
    'not updated: §5b'
assert 'trust' not in _insp.signature(solve_qp).parameters, \
    'solve_qp still has a trust param — you may have loaded the v10 QP cell or a stray patch'
assert 'trial_W' not in globals(), \
    'trial_W still defined (leftover from line search) — rerun top to bottom'
assert 'HP_LADDER' not in globals(), 'HP_LADDER still defined'
assert _status_ok('solved inaccurate') and _status_ok('solved') \
    and not _status_ok('primal infeasible'), '_status_ok behaves incorrectly'
_rr = _insp.getsource(run_repair)
assert 'LINE_SEARCH' not in _rr and 'reject' not in _rr, \
    'run_repair is not the paper-faithful version'
assert 'Algorithm 1' in _rr, 'run_repair is not the paper-faithful version'
assert 'curate_repair_set' in globals() and 'add_baselines' in globals(), \
    'not updated: curate_repair_set/add_baselines'
assert 'run_warp_matrix' in globals() and 'run_baseline_matrix' in globals(), \
    'not updated: run_warp_matrix/run_baseline_matrix'
assert "MARGIN_H'] - GAP_TOL" in _insp.getsource(full_eval), \
    'full_eval Cert 2 missing tolerance'

assert torch.backends.cuda.matmul.allow_tf32 is False, \
    'TF32 still enabled — Restart session, then rerun top to bottom'
assert os.environ.get('CUBLAS_WORKSPACE_CONFIG') == ':4096:8', \
    'CUBLAS_WORKSPACE_CONFIG not in effect — Restart session, then rerun top to bottom'

print(f'✓ PATCH_VERSION={PATCH_VERSION}')
print(f'OK: ALGO={ALGO_TAG} (full-step, no line search, no trust region, no HP ladder)')
print(f'✓ RUN_MODE={RUN_MODE}  FORCE_BUILTIN_OSQP={FORCE_BUILTIN_OSQP}')
print(f'✓ TF32=off  cudnn.deterministic=True  '
      f'OMP_NUM_THREADS={os.environ.get("OMP_NUM_THREADS")}')
print(f'OK: Cert 2 has GAP_TOL  BASELINE_N_SEEDS={BASELINE_N_SEEDS}')
print('OK: version check passed. Proceed to "Stage 1".')

## §11. Manual mode (single-group debug / step-through)

For inspecting one group or intermediate values, uncomment and run the cells below piece by piece. `debug_numpy_vs_torch` is a **required sanity check before a real run**: it verifies the numpy closed-form head matches torch logits (this is what catches a wrong repair-layer block selection).

In [ ]:
# Debug utilities
def debug_numpy_vs_torch(loaded, n=6, tol=1e-2):
    tok, model, adapter = loaded['tok'], loaded['model'], loaded['adapter']
    probes = GSN_PROBES[loaded['task']][:n]
    max_len = TASK_CONFIG[loaded['task']]['max_len']
    print(f'{"text":<44} {"torch_gap":>10} {"np_gap":>10} {"diff":>9}')
    print('-' * 78)
    max_d, n_ok = 0.0, 0
    for t in probes:
        v, ctx, tp = adapter.build_sample_ctx(model, tok, t, loaded['head_static'], max_len)
        enc = tok(t, return_tensors='pt', truncation=True, max_length=max_len)
        with torch.no_grad():
            lg = model(**{k: x.to(DEVICE) for k, x in enc.items()}).logits[0]
        tg = float(lg[1] - lg[0])
        g  = forward_gap_np(loaded['W0'], v, ctx)
        d  = abs(tg - g); max_d = max(max_d, d)
        n_ok += ((1 if g > 0 else 0) == tp)
        print(f'{t[:42]:<44} {tg:>10.4f} {g:>10.4f} {d:>9.5f}')
    ok = (max_d < tol) and (n_ok == len(probes))
    print(f'sign agreement {n_ok}/{len(probes)}  |  max |torch−np| = {max_d:.2e}'
          + ('  OK' if ok else f'  FAIL (value must be < {tol})'))
    return ok

print('Manual-mode utilities ready.')

## §12. Cleanup / maintenance (optional)

Removes known-broken stale artifacts.

In [ ]:
# Cleanup (optional)
import glob as _glob

_stale_result = [
    'result_distilgpt2_sst2.pkl',
    'result_pythia-160m_sst2.pkl', 'result_pythia-160m_rte.pkl',
    'result_pythia-410m_sst2.pkl', 'result_pythia-410m_rte.pkl',
]
_stale_gsnft = [
    'pythia-160m_sst2_gsnft.pt', 'pythia-160m_rte_gsnft.pt',
    'pythia-410m_sst2_gsnft.pt', 'pythia-410m_rte_gsnft.pt',
]
_stale_cache = [
    'cache_pythia-160m_sst2.pkl', 'cache_pythia-160m_rte.pkl',
    'cache_pythia-410m_sst2.pkl', 'cache_pythia-410m_rte.pkl',
]
_stale_qpckpt_globs = [
    'qp_ckpt_distilgpt2_sst2_*',
    'qp_ckpt_pythia-160m_*', 'qp_ckpt_pythia-410m_*',
]

n_removed = 0
for f in _stale_result + _stale_cache:
    p = f'{RESULT_DIR}/{f}'
    if os.path.exists(p):
        os.remove(p); n_removed += 1; print('removed', p)
for f in _stale_gsnft:
    p = f'{BASE}/ckpt_decoder/{f}'
    if os.path.exists(p):
        os.remove(p); n_removed += 1; print('removed', p)
for pat in _stale_qpckpt_globs:
    for p in _glob.glob(f'{RESULT_DIR}/{pat}'):
        os.remove(p); n_removed += 1; print('removed', p)

print(f'\nCleanup done, removed {n_removed} file(s).'
      f'\ndistilgpt2×sst2 task-FT / GSN-FT kept (not broken, no need to retrain).'
      f'\npythia task-FT kept, GSN-FT cleared → rerun will use KAPPA_TARGET to decide if GSN-FT is needed.')

In [ ]:
print(BASE)
print(os.listdir(BASE))

## Stage 1 — WARP (six groups, no baselines)

Runs `RUN_MATRIX` in order: `gpt2/sst2` → `gpt2/rte` → `gpt2-medium/sst2` → `gpt2-medium/rte` → `qwen2.5-0.5b/sst2` → `qwen2.5-0.5b/rte`.

After a disconnect, just rerun this cell to resume (`RUN_MODE='resume'`; Algorithm 1 is deterministic, so resuming from a checkpoint gives the same W as running straight through).

In [ ]:
# Resume-progress inventory
import os, glob, pickle, time

ALL6 = [('gpt2','sst2'), ('gpt2','rte'),
        ('gpt2-medium','sst2'), ('gpt2-medium','rte'),
        ('qwen2.5-0.5b','sst2'), ('qwen2.5-0.5b','rte')]

print(f'{"group":<24}{"WARP":<8}{"baseline":<10}{"StageA":<8}{"GSN":<6}'
      f'{"cache":<7}{"QP ckpt":<12}status')
print('=' * 104)
todo_warp, todo_base = [], []
for mk, tk in ALL6:
    rp  = f'{RESULT_DIR}/result_{mk}_{tk}.pkl'
    cp  = f'{RESULT_DIR}/cache_{mk}_{tk}.pkl'
    sa  = f'{CKPT_DIR}/{mk}_{tk}_taskft'
    gs  = f'{CKPT_DIR}/{mk}_{tk}_gsnft.pt'
    reg = MODEL_REGISTRY[mk]
    has_sa = bool(reg['task_ckpt'].get(tk)) or os.path.isdir(sa)
    iters  = sorted(glob.glob(f'{RESULT_DIR}/qp_ckpt_{mk}_{tk}_*_iter*.pkl'))
    fin    = glob.glob(f'{RESULT_DIR}/qp_ckpt_{mk}_{tk}_*_FINAL.pkl')
    last   = os.path.basename(iters[-1]).split('_iter')[-1][:4] if iters else '-'

    warp, base, note = '✗', '✗', ''
    if os.path.exists(rp):
        try:
            with open(rp, 'rb') as f: R = pickle.load(f)
            conv = R.get('converged')
            warp = '✓' if conv else '△'
            note = (f"conv={conv} iters={len(R.get('qp_log',[]))} "
                    f"W~={R.get('repro',{}).get('W_repaired_hash')}")
            if R.get('baselines'):
                b = R['baselines']
                base = '✓' + ('' if not R.get('baselines_stale') else '(stale)')
                note += f"  base={list(b.keys())}"
            else:
                todo_base.append((mk, tk))
        except Exception as e:
            warp, note = '?', f'read failed {type(e).__name__}'
    else:
        todo_warp.append((mk, tk))
        if iters:
            note = f'QP interrupted → can resume from iter {int(last)}'
        elif has_sa:
            note = 'Stage-A done, QP not started'
        else:
            note = 'not started at all (needs Stage-A FT)'

    print(f'{mk+"/"+tk:<24}{warp:<8}{base:<10}'
          f'{"✓" if has_sa else "✗":<8}{"✓" if os.path.exists(gs) else "✗":<6}'
          f'{"✓" if os.path.exists(cp) else "✗":<7}'
          f'{(last if iters else ("FINAL" if fin else "-")):<12}{note}')

print('\n' + '=' * 104)
print(f'WARP still to run     : {todo_warp}')
print(f'baseline still to run : {todo_base}')

print('\n5 most recently written files:')
fs = sorted(glob.glob(f'{RESULT_DIR}/*') + glob.glob(f'{CKPT_DIR}/*'),
            key=os.path.getmtime, reverse=True)[:5]
for p in fs:
    print(f'  {time.strftime("%H:%M:%S", time.localtime(os.path.getmtime(p)))}  '
          f'{os.path.basename(p)}')

In [ ]:
warp_res = run_warp_matrix()

## Stage 2 — Baselines (optional, run after Stage 1)

LoRA and Full FT, using the same repair set and starting point as WARP (verified via `W(0) hash`). Skip this cell if you don't want baselines — the summary table still shows the WARP rows.

In [ ]:
base_res = run_baseline_matrix()

## Summary

Reads `result_*.pkl`; prints WARP only if baselines are missing. Outputs the main table + per-attack breakdown + reproducibility/convergence table, and saves `{RESULT_DIR}/summary_matrix.md`.

In [ ]:
all_res, winners = print_summary()

In [ ]:
# Original-model evaluation (no GSN-FT)
import os, json, gc, pickle
import numpy as np
import torch

def load_stage_a_only(model_key, task):
    reg     = MODEL_REGISTRY[model_key]
    adapter = ADAPTERS[reg['family']]
    tcfg    = TASK_CONFIG[task]
    taskft_dir = f'{CKPT_DIR}/{model_key}_{task}_taskft'

    ckpt = reg['task_ckpt'].get(task)
    if ckpt:
        src = ckpt
    elif os.path.isdir(taskft_dir):
        meta_p = f'{taskft_dir}/ft_meta.json'
        done = tcfg['ft_epochs']
        if os.path.exists(meta_p):
            with open(meta_p) as f:
                done = json.load(f).get('epochs_done', 0)
        if done < tcfg['ft_epochs']:
            raise RuntimeError(
                f'{model_key}/{task}: Stage-A only completed {done}/{tcfg["ft_epochs"]} '
                f'epochs. This cell does not train — run run_warp_matrix() first to finish it.')
        src = taskft_dir
    else:
        raise RuntimeError(
            f'{model_key}/{task}: Stage-A artifact not found ({taskft_dir}). '
            f'Run run_warp_matrix() first.')

    print(f'[orig] {model_key}/{task} ← {src}  (no GSN-FT)')
    tok = AutoTokenizer.from_pretrained(reg['hf_id'])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForSequenceClassification.from_pretrained(
        src, num_labels=2, dtype=torch.float32).to(DEVICE)
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tok.pad_token_id
    model.eval()

    head_static = adapter.build_head_static(model)
    W_orig = adapter.weight_np(model)
    print(f'  W_orig hash = {arr_hash(W_orig)}  ||W||₂={np.linalg.norm(W_orig, 2):.4f}')
    return {'model': model, 'tok': tok, 'adapter': adapter,
            'W0': W_orig, 'head_static': head_static,
            'model_key': model_key, 'task': task,
            'v0_key': f'{model_key}:{task}'}

def eval_original(model_key, task, force=False):
    cpath = f'{RESULT_DIR}/cache_ORIG_{model_key}_{task}.pkl'
    L = load_stage_a_only(model_key, task)

    if os.path.exists(cpath) and not force:
        print(f'  loading existing original-model cache ← {cpath}')
        with open(cpath, 'rb') as f: D = pickle.load(f)
    else:
        print('  rebuilding cache with original weights (v unchanged, ctx needs recompute since score head differs)...')
        D = build_cache(L)
        with open(cpath, 'wb') as f: pickle.dump(D, f)
        print(f'  [saved] {cpath}')

    W = L['W0']
    per_atk = {m: eval_acc(c, W) for m, c in D['test_caches'].items()}
    out = {
        'W_orig_hash': arr_hash(W),
        'clean_acc_gold': eval_acc(D['clean_set'], W),
        'atk_per_method': per_atk,
        'atk_macro': float(np.mean(list(per_atk.values()))),
        'atk_micro': eval_acc([s for c in D['test_caches'].values() for s in c], W),
        'n_clean': len(D['clean_set']),
        'n_atk': sum(len(c) for c in D['test_caches'].values()),
        'n_dev_wrong': len(D['repair_set']),
    }
    print(f"  clean(gold)={out['clean_acc_gold']:.1f}%  "
          f"AtkGen macro={out['atk_macro']:.1f}%  micro={out['atk_micro']:.1f}%  "
          f"dev wrong {out['n_dev_wrong']} examples")
    del L['model']; del L
    gc.collect(); torch.cuda.empty_cache()
    return out

def original_matrix(force=False):
    orig = {}
    for mk, tk in RUN_MATRIX:
        print(f'\n{"="*74}\n  ORIGINAL  {mk} × {tk}\n{"="*74}')
        try:
            orig[(mk, tk)] = eval_original(mk, tk, force=force)
        except Exception as e:
            import traceback as _tb
            msg = f'{type(e).__name__}: {e}'
            _tb.clear_frames(e.__traceback__); del e
            gc.collect(); torch.cuda.empty_cache()
            print(f'[ERROR] {mk}/{tk}: {msg}')

    print(f'\n\n{"="*96}')
    print('Original model vs W(0) vs WARP — all against gold label (differs from the main table General column)')
    print('=' * 96)
    print(f'{"model/task":<22}{"stage":<20}{"Clean(gold)":>13}'
          f'{"AtkGen macro":>14}{"AtkGen micro":>14}')
    rows_md = ['| Model | Task | Stage | Clean (gold) | AtkGen macro | AtkGen micro |',
               '|---|---|---|---|---|---|']
    for (mk, tk), o in orig.items():
        rp = f'{RESULT_DIR}/result_{mk}_{tk}.pkl'
        R = pickle.load(open(rp, 'rb')) if os.path.exists(rp) else None
        stages = [('Original (Stage-A)', o['clean_acc_gold'],
                   o['atk_macro'], o['atk_micro'])]
        if R:
            cp = f'{RESULT_DIR}/cache_{mk}_{tk}.pkl'
            if os.path.exists(cp):
                D0 = pickle.load(open(cp, 'rb'))
                pool = [s for c in D0['test_caches'].values() for s in c]
                for lab, W in [('W(0) post-GSN-FT', None),
                               ('WARP repaired', R['W_repaired'])]:
                    if W is None:
                        cl = 100.0 * sum(s['correct'] for s in D0['clean_set']) \
                             / len(D0['clean_set'])
                        pm = {m: 100.0 * sum(s['correct'] for s in c) / len(c)
                              for m, c in D0['test_caches'].items()}
                        mi = 100.0 * sum(s['correct'] for s in pool) / len(pool)
                    else:
                        cl = eval_acc(D0['clean_set'], W)
                        pm = {m: eval_acc(c, W) for m, c in D0['test_caches'].items()}
                        mi = eval_acc(pool, W)
                    stages.append((lab, cl, float(np.mean(list(pm.values()))), mi))
        for lab, cl, ma, mi in stages:
            print(f'{mk+"/"+tk:<22}{lab:<20}{cl:>12.1f}%{ma:>13.1f}%{mi:>13.1f}%')
            rows_md.append(f'| {mk} | {tk} | {lab} | {cl:.1f}% | {ma:.1f}% | {mi:.1f}% |')
        print('-' * 96)

    print(f'\n{"="*96}\nOriginal model per-attack (gold label)\n{"="*96}')
    methods = sorted({m for o in orig.values() for m in o['atk_per_method']})
    print(f'{"model/task":<22}' + ''.join(f'{m:>13}' for m in methods) + f'{"macro":>9}')
    rows_md += ['', '### Original model per-attack (gold)', '',
                '| Model | Task | ' + ' | '.join(methods) + ' | Macro |',
                '|---' * (len(methods) + 3) + '|']
    for (mk, tk), o in orig.items():
        g = o['atk_per_method']
        print(f'{mk+"/"+tk:<22}'
              + ''.join(f'{g.get(m, float("nan")):>12.1f}%' for m in methods)
              + f'{o["atk_macro"]:>8.1f}%')
        rows_md.append(f'| {mk} | {tk} | '
                       + ' | '.join(f'{g.get(m, float("nan")):.1f}%' for m in methods)
                       + f' | {o["atk_macro"]:.1f}% |')

    with open(f'{RESULT_DIR}/original_models.md', 'w') as f:
        f.write('\n'.join(rows_md) + '\n')
    with open(f'{RESULT_DIR}/original_models.pkl', 'wb') as f:
        pickle.dump(orig, f)
    print(f'\n[saved] {RESULT_DIR}/original_models.md')
    print(f'[saved] {RESULT_DIR}/original_models.pkl')
    return orig

orig_res = original_matrix()

In [ ]:
# Final consolidated table
import os, pickle, glob
import numpy as np

ALL6 = [('gpt2','sst2'), ('gpt2','rte'),
        ('gpt2-medium','sst2'), ('gpt2-medium','rte'),
        ('qwen2.5-0.5b','sst2'), ('qwen2.5-0.5b','rte')]
KEYS = [k for k in ALL6 if os.path.exists(f'{RESULT_DIR}/result_{k[0]}_{k[1]}.pkl')]

FULLFT_PARAMS = {'gpt2': 124_441_344, 'gpt2-medium': 354_825_216}

_op = f'{RESULT_DIR}/original_models.pkl'
ORIG = pickle.load(open(_op, 'rb')) if os.path.exists(_op) else {}
print(f'Loaded {len(KEYS)} result group(s), {len(ORIG)} original-eval group(s)\n')

def _agree_with_w0(mk, tk):
    cp, op = f'{RESULT_DIR}/cache_{mk}_{tk}.pkl', f'{RESULT_DIR}/cache_ORIG_{mk}_{tk}.pkl'
    if not (os.path.exists(cp) and os.path.exists(op)):
        return None, None
    try:
        D0 = pickle.load(open(cp, 'rb')); DO = pickle.load(open(op, 'rb'))
        a, b = D0['clean_set'], DO['clean_set']
        if len(a) != len(b) or any(x.get('text') != y.get('text')
                                   for x, y in zip(a[:50], b[:50])):
            return None, None
        agree = 100.0 * sum(1 for x, y in zip(a, b) if x['pred'] == y['pred']) / len(a)
        w0_gold = 100.0 * sum(s['correct'] for s in a) / len(a)
        return agree, w0_gold
    except Exception:
        return None, None

def _fmt(v, suf='%'):
    return f'{"—":>6} ' if v is None else f'{v:6.1f}{suf}'

print('=' * 104)
print('Main table — Repair / Remain / General compare against W(0) predictions; AtkGen uses gold label')
print('=' * 104)
print(f'{"setting":<22}{"method":<14}{"Repair":>9}{"Remain":>9}{"General":>9}'
      f'{"AtkGen":>9}   {"params changed":>16}  note')
print('-' * 104)

md = ['| Setting | Method | Repair | Remain | General | AtkGen | Params changed |',
      '|---|---|---|---|---|---|---|']
MAIN = {}
for mk, tk in KEYS:
    R = pickle.load(open(f'{RESULT_DIR}/result_{mk}_{tk}.pkl', 'rb'))
    W, B = R['evals']['after'], (R.get('baselines') or {})
    lo, fu = B.get('lora'), B.get('full_ft')
    o = ORIG.get((mk, tk))
    agree, w0_gold = _agree_with_w0(mk, tk)
    d_out, d_in = R['W_repaired'].shape
    n_warp = R['HP']['RANK'] * d_in

    rows = []
    if o:
        rows.append(('Original', None, None, None, o['atk_macro'], 0,
                     'Stage-A, before GSN-FT / repair'))
    rows.append(('GSN-FT W(0)', 0.0, 100.0, 100.0,
                 R['evals']['before']['gen_mean'], None, 'QP starting point'))
    if lo:
        r_l = max(1, round(R['HP']['RANK'] * d_in / (d_in + d_out)))
        rows.append(('LoRA', lo['repair_acc'], lo['remain_acc'], lo['clean_acc'],
                     lo['gen_mean'], r_l * (d_in + d_out), f'same-layer, r={r_l}'))
    if fu:
        rows.append(('Full FT', fu['repair_acc'], fu['remain_acc'], fu['clean_acc'],
                     fu['gen_mean'], FULLFT_PARAMS.get(mk), 'all layers'))
    rows.append(('WARP (ours)', W['repair_acc'], W['remain_acc'], W['clean_acc'],
                 W['gen_mean'], n_warp,
                 f"rank={R['HP']['RANK']}, conv={R['converged']}"))

    for name, rp, rm, ge, at, npar, note in rows:
        ps = ('—' if npar is None else
              ('0' if npar == 0 else f'{npar:,}'))
        print(f'{mk+"/"+tk:<22}{name:<14}{_fmt(rp)}{_fmt(rm)}{_fmt(ge)}{_fmt(at)}'
              f'   {ps:>16}  {note}')
        md.append(f'| {mk}/{tk} | {name} | '
                  + ' | '.join('—' if v is None else f'{v:.1f}%' for v in (rp, rm, ge, at))
                  + f' | {ps} |')
    MAIN[(mk, tk)] = (R, o, agree, w0_gold)
    print('-' * 104)

print("\nOriginal shows '—' for Repair / Remain / General because:")
print("  - repair_set / remain_set are defined by W(0)'s predictions (wrong / correct),")
print("    which don't carry meaning for a model that hasn't gone through GSN-FT.")
print("  - clean_set is filtered in build_cache to samples 'this model got right':")
print("      clean_set = [s for s in clean_all if s['correct']][:n_cln]")
print("    so gold clean accuracy is always 100% — uninformative;")
print("    and the original model and W(0) select different subset members, so not directly comparable.")
print("  - AtkGen is the exception: test_caches includes all AdvGLUE samples, unfiltered,")
print("    so every row uses the same basis (gold label) and can be compared directly.")

if ORIG:
    print("\n" + "=" * 104)
    print("Original model vs W(0): change from GSN-FT (only AtkGen is comparable)")
    print("=" * 104)
    print(f'{"setting":<22}{"Orig AtkGen":>13}{"W(0) AtkGen":>13}{"Δ":>9}'
          f'{"κ pre":>9}{"κ post":>9}{"orig dev wrong":>15}{"W(0) |S|":>10}')
    for (mk, tk), (R, o, agree, w0_gold) in MAIN.items():
        if not o:
            continue
        ob, wb = o["atk_macro"], R["evals"]["before"]["gen_mean"]
        g = R["gsn"]
        print(f'{mk+"/"+tk:<22}{ob:>12.1f}%{wb:>12.1f}%{wb-ob:>+8.1f}%'
              f'{g["pre"]:>9.4f}{g["post"]:>9.4f}'
              f'{o["n_dev_wrong"]:>15}{R["curation"]["n_total"]:>10}')
    print("\n  κ pre == κ post means GSN-FT was skipped (κ already reached KAPPA_TARGET);")
    print("  in that case the original model equals W(0), so both AtkGen columns match.")

methods = sorted({m for (mk, tk), (R, *_ ) in MAIN.items()
                  for m in R['evals']['after']['gen']})
print('\n' + '=' * 104)
print('PER-ATTACK(gold label)')
print('=' * 104)
print(f'{"setting":<22}{"method":<14}' + ''.join(f'{m:>12}' for m in methods)
      + f'{"macro":>9}')
md += ['', '### Per-attack (gold label)', '',
       '| Setting | Method | ' + ' | '.join(methods) + ' | Macro |',
       '|---' * (len(methods) + 3) + '|']
for (mk, tk), (R, o, *_ ) in MAIN.items():
    B = R.get('baselines') or {}
    series = []
    if o: series.append(('Original', o['atk_per_method'], o['atk_macro']))
    series.append(('GSN-FT W(0)', R['evals']['before']['gen'],
                   R['evals']['before']['gen_mean']))
    for lab, k in [('LoRA', 'lora'), ('Full FT', 'full_ft')]:
        if B.get(k):
            series.append((lab, B[k].get('gen', {}), B[k]['gen_mean']))
    series.append(('WARP (ours)', R['evals']['after']['gen'],
                   R['evals']['after']['gen_mean']))
    for lab, g, mean in series:
        print(f'{mk+"/"+tk:<22}{lab:<14}'
              + ''.join(f'{g.get(m, float("nan")):>11.1f}%' for m in methods)
              + f'{mean:>8.1f}%')
        md.append(f'| {mk}/{tk} | {lab} | '
                  + ' | '.join(f'{g.get(m, float("nan")):.1f}%' for m in methods)
                  + f' | {mean:.1f}% |')
    print('-' * 104)

print('\n' + '=' * 104)
print('DIAGNOSTIC LOG')
print('=' * 104)
for (mk, tk), (R, o, agree, w0_gold) in MAIN.items():
    W, Bf = R['evals']['after'], R['evals']['before']
    Ws = R['evals'].get('ref')
    B = R.get('baselines') or {}
    g = R['gsn']; c = R['curation']; rp = R['repro']
    d_out, d_in = R['W_repaired'].shape
    print(f'\n■ {mk} × {tk}')
    print(f'  HP           {R["HP"]}')
    print(f'  repair layer ({d_out}, {d_in})  decision vars = {R["HP"]["RANK"]}×{d_in}'
          f' = {R["HP"]["RANK"]*d_in:,}')
    print(f'  GSN κ        {g["pre"]:.4f} → {g["post"]:.4f}  (best step={g["best_step"]})')
    print(f'  curation     kept {c["n_used"]}/{c["n_total"]}  floor={c["floor"]:.2f}'
          f'  gap scale={c["gap_scale"]:.2f}')
    if c['dropped_init_gaps']:
        print(f'               dropped gaps={np.round(c["dropped_init_gaps"], 2).tolist()}')
    print(f'  set sizes    |S|={R["set_sizes"]["S"]}  |P|={R["set_sizes"]["P"]}  '
          f'|Gen|={R["set_sizes"]["Gen"]}  |AtkGen|={R["set_sizes"]["AtkGen"]}')
    print(f'  converged    {R["converged"]}  iters={len(R["qp_log"])}  '
          f'{R["elapsed"]/60:.1f} min')
    if R['qp_log']:
        l0, ln = R['qp_log'][0], R['qp_log'][-1]
        print(f'  ‖W‖₂         {l0["spec_norm"]:.2f} → {ln["spec_norm"]:.2f}'
              f'  ({ln["spec_norm"]/l0["spec_norm"]:.3f}x)'
              f'   ‖dW‖_F first={l0.get("dW_norm", float("nan")):.3f}'
              f' last={ln.get("dW_norm", float("nan")):.3f}')
    if Ws:
        print(f'  W0-swap(ref) repair={Ws["repair_acc"]:.1f}%  '
              f'remain={Ws["remain_acc"]:.1f}%  General={Ws["clean_acc"]:.1f}%  '
              f'AtkGen={Ws["gen_mean"]:.1f}%   (ctx is post-GSN-FT, reference only)')
    print(f'  cert (WARP)  Cert1 {W["n_cert1"]}/{R["set_sizes"]["S"]}   '
          f'Cert2 {W["n_cert2"]}/{R["set_sizes"]["P"]}   '
          f'labels kept {W["n_label_keep"]}/{R["set_sizes"]["P"]}')
    eps = np.asarray(W['eps_star'])
    print(f'  Cert3        kind={W["cert3_kind"]}  ε* med={np.median(eps):.2e} '
          f'max={eps.max():.2e}  (float32 eps={np.finfo(np.float32).eps:.2e})'
          f'  → valid {int((eps > np.finfo(np.float32).eps).sum())}/{len(eps)}')
    if B.get('lora'):
        L = B['lora']
        print(f'  cert (LoRA)  Cert1 {L.get("n_cert1","?")}/{R["set_sizes"]["S"]}   '
              f'Cert2 {L.get("n_cert2","?")}/{R["set_sizes"]["P"]}')
    if B.get('full_ft'):
        print(f'  cert (FullFT) N/A — changes all layers, the fixed-head decomposition does not hold, '
              f'this framework\'s certificate does not apply')
    print(f'  ‖ΔW‖_F       WARP={W["dW_fro"]:.2f}'
          + (f"   LoRA={B['lora'].get('dW_fro', float('nan')):.2f}" if B.get('lora') else '')
          + '   (relative ratio to ||W||_F is in each full_eval output)')
    print(f'  repro hash   W(0)={rp["W0_hash"]}  W~={rp["W_repaired_hash"]}'
          + (f"  W_orig={o['W_orig_hash']}" if o else ''))
    if R.get('baselines_mismatch'):
        print(f'  Warning: baseline and WARP starting points differ, not comparable')

print('\n' + '=' * 104)
print('ENVIRONMENT / COMPLETENESS')
print('=' * 104)
_env = next(iter(MAIN.values()))[0].get('env', {})
for k, v in _env.items(): print(f'  {k:<14}{v}')
print(f'  algo          {next(iter(MAIN.values()))[0].get("algo")}')
missing = [f'{a}/{b}' for a, b in ALL6 if (a, b) not in MAIN]
nobase  = [f'{a}/{b}' for (a, b), (R, *_ ) in MAIN.items() if not R.get('baselines')]
noorig  = [f'{a}/{b}' for (a, b) in MAIN if (a, b) not in ORIG]
print(f'\n  missing WARP      : {missing or "none"}')
print(f'  missing baseline  : {nobase or "none"}')
print(f'  missing original  : {noorig or "none"}')
nonconv = [f'{a}/{b}' for (a, b), (R, *_ ) in MAIN.items() if not R['converged']]
print(f'  not converged     : {nonconv or "none — all gap_sat full and remain=100%"}')

with open(f'{RESULT_DIR}/FINAL_TABLE.md', 'w') as f:
    f.write('\n'.join(md) + '\n')
print(f'\n[saved] {RESULT_DIR}/FINAL_TABLE.md')

## Rerun a single group (debug)

```python
r = run_one('gpt2', 'sst2', force=True)      # WARP only
add_baselines('gpt2', 'sst2', force=True)    # baseline only
```